# Public-screen gene similarity → borrowed H1 knockdown effects

Learns which genes behave alike when knocked down from public Perturb-seq screens (Replogle 2022 K562 genome-wide and RPE1; Nadig 2025 HepG2 and Jurkat), uses that similarity to weight the real effects of the *other* H1 knockdowns, and scores the result on H1 with the pseudobulk Pearson scorer and Arc's cell-eval, next to the mean-effect (0.19) and chance baselines.

Everything downloaded here is public: no VCC key is needed. Outputs go to Google Drive.

**Runtime:** a high-RAM CPU runtime is enough (about 25 GB disk for the 2025 support set). Expect roughly 15 min to download, 15 min for the H1 pseudobulk, and 10–20 min for cell-eval.

In [ ]:
!pip -q install anndata h5py cell-eval
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/vcc/public-screens'   # change if you like
WORK = '/content/vcc'; DATA = '/content/data'
for d in (OUT, WORK, DATA): os.makedirs(d, exist_ok=True)
os.chdir(WORK)

## Pipeline scripts
The same files that live in the project's shared folder (`logo-shap/v2/` and `public-screens/`), written into `/content/vcc`. The screen scripts look for the v2 helpers in their own folder here.

In [ ]:
%%writefile grn_ridge.py
"""All-genes ridge GRN from control cells, with grouped CV and stability stats.

The fast path for stage 1 of grn_linear_model.pdf. Instead of fitting one regression per
target gene, every "gene ~ all other genes" ridge comes out of one matrix inverse:
with standardized expression Z and P = (Z'Z/n + lam*I)^-1, the coefficients predicting
gene i are  -P[:, i] / P[i, i]  (exactly what sklearn's Ridge gives per gene). One
eigendecomposition per fold makes every lam essentially free.

Per fold (GroupKFold over guides, or full leave-one-group-out):
  coef      regulator j -> target i, in log1p(CP10k) units (the slope, for propagation)
  shap      mean |SHAP| on held-out cells = |coef| * mean|x_j - mean_j| (exact for linear)
  r2        held-out R^2 for every target
Across folds:
  coef_mean, coef_std, sign_agree (fraction of folds whose sign matches the mean's),
  shap_share (each fold's |SHAP| column scaled to sum 1 per target, then averaged), r2.

Cell-state removal (--n-pcs, --covariates) regresses log library size, mitochondrial
fraction and the top principal components out of every gene before fitting, so edges
reflect residual co-variation rather than shared programs (cell cycle, quality,
differentiation).

    python grn_ridge.py --h5ad context_A.h5ad --group-col ntc_id --out grn_A.npz
"""

from __future__ import annotations

import argparse
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

LAMBDAS = (0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0)


def load(h5ad, group_col, min_frac=0.05):
    """log1p(CP10k) expression (dense, genes detected in >= min_frac of cells), plus covariates."""
    import anndata as ad

    a = ad.read_h5ad(h5ad)
    if "target_gene" in a.obs and (a.obs["target_gene"] != "non-targeting").any():
        a = a[(a.obs["target_gene"] == "non-targeting").to_numpy()].copy()
    X = sp.csr_matrix(a.X, dtype=np.float32)
    lib = np.asarray(X.sum(1)).ravel()
    mt = np.asarray(X[:, np.flatnonzero(np.char.startswith(a.var_names.to_numpy().astype(str), "MT-"))].sum(1)).ravel() / lib
    X = sp.diags((1e4 / lib).astype(np.float32)) @ X
    X.data = np.log1p(X.data)
    X.eliminate_zeros()
    keep = np.bincount(X.indices, minlength=X.shape[1]) / X.shape[0] >= min_frac
    covars = np.column_stack([np.log(lib), mt])
    return X[:, keep].toarray(), a.var_names.to_numpy()[keep], a.obs[group_col].astype(str).to_numpy(), covars


def remove_cell_state(X, covars, n_pcs=10, log=print):
    """Residualize every gene on covariates and the top n_pcs principal components."""
    parts = [np.ones((len(X), 1))]
    if covars is not None:
        parts.append(covars)
    if n_pcs:
        Xs = (X - X.mean(0)) / (X.std(0) + 1e-8)
        from sklearn.utils.extmath import randomized_svd
        u, s, _ = randomized_svd(Xs, n_components=n_pcs, random_state=0)
        parts.append(u * s)
        log(f"  removed {n_pcs} PCs (variance share {(s**2).sum() / (Xs**2).sum():.1%})")
    C = np.column_stack(parts)
    beta, *_ = np.linalg.lstsq(C, X, rcond=None)
    return (X - C @ beta).astype(np.float32)


def _edges_for_lambdas(G_eig, lam):
    L, V = G_eig
    P = (V / (L + lam)) @ V.T
    d = np.diag(P).copy()
    B = -P / d[None, :]
    np.fill_diagonal(B, 0.0)
    return B  # B[j, i]: standardized coef of regulator j predicting target i


def _r2(X_te, mu, sd, B_raw):
    pred = (X_te - mu) @ B_raw + mu
    ss_res = ((X_te - pred) ** 2).sum(0)
    ss_tot = ((X_te - mu) ** 2).sum(0)
    return 1 - ss_res / np.maximum(ss_tot, 1e-12)


def fit(X, groups, n_folds=5, lambdas=LAMBDAS, lam=None, log=print):
    from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

    n_genes = X.shape[1]
    splitter = LeaveOneGroupOut() if n_folds in (None, 0, "logo") else GroupKFold(n_splits=n_folds)
    coef_sum = np.zeros((n_genes, n_genes), dtype=np.float32)
    coef_sq = np.zeros((n_genes, n_genes), dtype=np.float32)
    pos = np.zeros((n_genes, n_genes), dtype=np.int32)
    share_sum = np.zeros((n_genes, n_genes), dtype=np.float32)
    shap_sum = np.zeros((n_genes, n_genes), dtype=np.float32)
    r2s, k = [], 0
    for k, (tr, te) in enumerate(splitter.split(X, groups=groups), start=1):
        t0 = time.perf_counter()
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-8
        Z = (X[tr] - mu) / sd
        G = (Z.T @ Z / len(tr)).astype(np.float64)
        del Z
        L, V = np.linalg.eigh(G)
        del G
        eig = (L.astype(np.float32), V.astype(np.float32))
        del L, V
        if lam is None:  # pick lam once, on the first fold, by mean held-out R^2
            scores = {}
            for l in lambdas:
                B = _edges_for_lambdas(eig, l) * (sd[None, :] / sd[:, None])
                scores[l] = float(np.mean(_r2(X[te], mu, sd, B)))
            lam = max(scores, key=scores.get)
            log("  lambda search (mean held-out R2): " + ", ".join(f"{l:g}={s:.3f}" for l, s in scores.items()))
        B = _edges_for_lambdas(eig, lam) * (sd[None, :] / sd[:, None])  # raw-unit slopes
        r2 = _r2(X[te], mu, sd, B)
        shap = np.abs(B) * np.abs(X[te] - mu).mean(0)[:, None]
        coef_sum += B
        coef_sq += B**2
        pos += B > 0
        shap_sum += shap
        share_sum += shap / np.maximum(shap.sum(0, keepdims=True), 1e-12)
        r2s.append(r2)
        log(f"  fold {k}: {len(te)} held-out cells, mean R2 {r2.mean():.3f}, median {np.median(r2):.3f} "
            f"({time.perf_counter() - t0:.0f}s)")
    coef_mean = coef_sum / k
    sign_agree = np.where(coef_mean > 0, pos / k, 1 - pos / k)
    np.fill_diagonal(sign_agree, 0)
    return {
        "coef_mean": coef_mean.astype(np.float32),
        "coef_std": np.sqrt(np.maximum(coef_sq / k - coef_mean**2, 0)).astype(np.float32),
        "sign_agree": sign_agree.astype(np.float32),
        "shap_mean": (shap_sum / k).astype(np.float32),
        "shap_share": (share_sum / k).astype(np.float32),
        "r2": np.mean(r2s, 0).astype(np.float32),
        "lam": lam,
        "n_folds": k,
    }


def top_edges(res, genes, per_target=20, min_sign_agree=0.8):
    """Long-format table of each target's strongest stable regulators."""
    rows = []
    share, coef, agree = res["shap_share"], res["coef_mean"], res["sign_agree"]
    for i, t in enumerate(genes):
        order = np.argsort(-share[:, i])
        order = [j for j in order if agree[j, i] >= min_sign_agree][:per_target]
        for rank, j in enumerate(order, 1):
            rows.append({"target": t, "regulator": genes[j], "rank": rank, "shap_share": share[j, i],
                         "coef": coef[j, i], "sign_agree": agree[j, i], "target_r2": res["r2"][i]})
    return pd.DataFrame(rows)


def run(h5ad, group_col, out, n_folds=5, n_pcs=10, covariates=True, min_frac=0.05, lam=None, log=print):
    t0 = time.perf_counter()
    X, genes, groups, covars = load(h5ad, group_col, min_frac)
    log(f"{X.shape[0]} cells x {X.shape[1]} genes, {len(np.unique(groups))} groups by {group_col!r}")
    if n_pcs or covariates:
        X = remove_cell_state(X, covars if covariates else None, n_pcs, log)
    res = fit(X, groups, n_folds, lam=lam, log=log)
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out, genes=genes.astype(str), **{k: v for k, v in res.items()})
    top_edges(res, genes).to_csv(out.with_suffix(".top_edges.csv"), index=False)
    log(f"lambda {res['lam']:g}, {res['n_folds']} folds, mean R2 {res['r2'].mean():.3f}; "
        f"wrote {out} and {out.with_suffix('.top_edges.csv').name} ({time.perf_counter() - t0:.0f}s)")
    return genes, res


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--h5ad", required=True)
    p.add_argument("--group-col", default="ntc_id")
    p.add_argument("--out", required=True, help="output .npz")
    p.add_argument("--folds", default="5", help="number of grouped folds, or 'logo' for leave-one-group-out")
    p.add_argument("--n-pcs", type=int, default=10, help="principal components to regress out (0 = none)")
    p.add_argument("--no-covariates", action="store_true", help="don't regress out library size / MT fraction")
    p.add_argument("--min-frac", type=float, default=0.05)
    p.add_argument("--lam", type=float, default=None, help="fix lambda instead of searching")
    a = p.parse_args()
    folds = 0 if a.folds == "logo" else int(a.folds)
    run(a.h5ad, a.group_col, a.out, folds, a.n_pcs, not a.no_covariates, a.min_frac, a.lam)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_h1.py
"""Simulate H1 knockdowns through a control-learned GRN and score against the real ones.

For each knocked-down gene k that is in the network, the knockdown's change propagates
through the edges (stage 2 of grn_linear_model.pdf, in its well-posed "change" form):

    d = e_k * d_k + W' d      (iterated; gene k stays clamped at d_k)

W[j, i] is the ridge slope of regulator j on target i (log1p CP10k units). Because the
largest |eigenvalue| of W is < 1 the iteration converges; `steps` caps it (1 = direct
targets only).

d_k (how far the knocked-down gene itself drops) is either the observed value (oracle)
or the median over all H1 knockdowns (what we'd have to assume zero-shot).

Scored per knockdown over the network's genes, excluding k itself:
  pearson_all     Pearson(predicted change, real change) over all genes
  pearson_top100  same, on the 100 genes the knockdown changed most
  sign_top50      fraction of the 50 most-changed genes whose direction is right

Baselines:
  mean_effect     the average real change over the other 149 knockdowns (no GRN)
  coexpression    marginal slope cov(i, k) / var(k) from controls (no conditioning)
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))
import grn_ridge  # noqa: E402


def propagate(W, idx, d_self, steps):
    """Change in every gene for each knockdown (columns), via clamped iteration."""
    n, p = W.shape[0], len(idx)
    D0 = np.zeros((n, p), dtype=np.float64)
    D0[idx, np.arange(p)] = d_self
    D = D0.copy()
    Wt = W.T.astype(np.float64)
    for _ in range(steps):
        D = D0 + Wt @ D
        D[idx, np.arange(p)] = d_self
    return D


def score(pred, true, k):
    mask = np.ones(len(true), bool)
    mask[k] = False
    p, t = pred[mask], true[mask]
    out = {}
    out["pearson_all"] = np.corrcoef(p, t)[0, 1] if p.std() > 0 else np.nan
    top = np.argsort(-np.abs(t))
    t100 = top[:100]
    out["pearson_top100"] = np.corrcoef(p[t100], t[t100])[0, 1] if p[t100].std() > 0 else np.nan
    t50 = top[:50]
    out["sign_top50"] = float(np.mean(np.sign(p[t50]) == np.sign(t[t50])))
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--grn", nargs="+", required=True, help="grn_ridge .npz files to evaluate")
    ap.add_argument("--pseudobulk", required=True)
    ap.add_argument("--controls", required=True, help="H1 controls .h5ad (for the co-expression baseline)")
    ap.add_argument("--out", required=True, help="output CSV of per-knockdown scores")
    a = ap.parse_args()

    pb = np.load(a.pseudobulk, allow_pickle=True)
    delta_all, pb_genes, perts = pb["delta"].astype(np.float64), list(pb["genes"]), pb["perts"]
    gi_all = {g: i for i, g in enumerate(pb_genes)}
    self_all = np.array([delta_all[i, gi_all[p]] for i, p in enumerate(perts)])
    d_fixed = float(np.median(self_all))
    print(f"median self-knockdown change {d_fixed:.3f} (log1p CP10k)")

    rows = []
    coexp_done = set()
    for path in a.grn:
        g = np.load(path, allow_pickle=True)
        genes = list(g["genes"])
        W = g["coef_mean"]
        name = Path(path).stem
        cols = np.array([gi_all[x] for x in genes])
        true = delta_all[:, cols].T                       # network genes x perts
        in_net = np.array([p in set(genes) for p in perts])
        idx = np.array([genes.index(p) for p in perts[in_net]])
        P = np.flatnonzero(in_net)
        T = true[:, P]
        print(f"{name}: {len(genes)} genes, {len(P)}/{len(perts)} knockdowns in network")

        preds = {}
        for steps in (1, 3, 20):
            preds[f"grn_oracle_{steps}step"] = propagate(W, idx, T[idx, np.arange(len(P))], steps)
        preds["grn_fixed_20step"] = propagate(W, idx, np.full(len(P), d_fixed), 20)

        # Leave-one-out mean effect over the other knockdowns, alone and plus the GRN shift.
        tot = true.sum(1, keepdims=True)
        mean_eff = (tot - true)[:, P] / (true.shape[1] - 1)
        preds["mean_effect+grn"] = mean_eff + preds["grn_oracle_20step"]
        if "mean_effect" not in coexp_done:
            preds["mean_effect"] = mean_eff
            # Co-expression: marginal slope of every gene on k in the raw (no state removal) controls.
            X, cgenes, _, _ = grn_ridge.load(a.controls, "batch", 0.0)
            ci = {x: i for i, x in enumerate(cgenes)}
            Xg = X[:, [ci[x] for x in genes]].astype(np.float64)
            Xg -= Xg.mean(0)
            Xk = Xg[:, idx]
            slope = (Xg.T @ Xk) / (Xk**2).sum(0)              # genes x perts
            preds["coexpression"] = slope * T[idx, np.arange(len(P))]
            del X, Xg
            coexp_done.add("mean_effect")

        for method, M in preds.items():
            label = method if method in ("mean_effect", "coexpression") else f"{name}:{method}"
            for c, pi in enumerate(P):
                r = score(M[:, c], T[:, c], idx[c])
                rows.append({"method": label, "pert": perts[pi], "effect_size": np.abs(T[:, c]).mean(), **r})

    df = pd.DataFrame(rows)
    df.to_csv(a.out, index=False)
    strong = df.effect_size >= df.groupby("method").effect_size.transform(lambda s: s.quantile(0.75))
    summ = df.groupby("method")[["pearson_all", "pearson_top100", "sign_top50"]].mean()
    summ["pearson_all_strongest25%"] = df[strong].groupby("method").pearson_all.mean()
    print(summ.round(3).sort_values("pearson_all", ascending=False).to_string())
    summ.round(4).to_csv(Path(a.out).with_suffix(".summary.csv"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile celleval_h1.py
"""Score candidate submissions on real H1 knockdowns with Arc's official cell-eval metrics.

Every method builds a prediction the way a real submission would: sample 400 control
cells per knockdown, shift each cell by a predicted change in log1p(CP10k) space, and
convert back to integer counts. Methods:

  ctrl_sampled   resampled control cells, no change (the "nothing happens" baseline)
  ctrl_mean      every cell = rounded control mean (the repo's baseline_control_mean)
  mean_effect    control cells + the leave-one-out mean effect of the other knockdowns
  network        control cells + the control-cell GRN's propagated change (fixed self-drop)
  oracle         control cells + the knockdown's real pseudobulk change (an upper bound
                 for "shift the controls" predictions; not achievable zero-shot)

The real data is the test .h5ad from h1_extract_perts.py. cell-eval normalizes raw
counts itself and computes DE with pdex. Metrics in the "vcc" profile: mae (lower is
better), discrimination_score_l1 and overlap_at_N (higher is better).

    python celleval_h1.py --real h1_real_test.h5ad --pseudobulk h1_pseudobulk.npz \
        --grn grn_h1_pc0.npz --out celleval_results/
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

sys.path.insert(0, str(Path(__file__).parent))
from eval_h1 import propagate  # noqa: E402

SELF_DROP = -0.633  # H1 median change of the knocked-down gene itself, log1p(CP10k)


def to_lognorm(X):
    X = sp.csr_matrix(X, dtype=np.float32)
    lib = np.asarray(X.sum(1)).ravel()
    Xn = sp.diags((1e4 / np.maximum(lib, 1)).astype(np.float32)) @ X
    Xn.data = np.log1p(Xn.data)
    return Xn, lib


def shifted_counts(ctrl_log, lib, delta):
    """Apply a per-gene log change to control cells and return integer counts."""
    out = np.asarray(ctrl_log.todense(), dtype=np.float32)
    out += delta[None, :].astype(np.float32)
    np.clip(out, 0, None, out=out)
    counts = np.expm1(out) * (lib / 1e4)[:, None]
    return sp.csr_matrix(np.rint(counts).astype(np.float32))


def build_pred(real, deltas, cells_per_pert, rng):
    """deltas: dict pert -> change vector over real.var_names (zeros for 'no change')."""
    ctrl_mask = (real.obs["target"] == "non-targeting").to_numpy()
    ctrl = real[ctrl_mask]
    ctrl_log, lib = to_lognorm(ctrl.X)
    blocks, obs = [ctrl.X.copy()], [pd.DataFrame({"target": ["non-targeting"] * ctrl.n_obs})]
    for pert, d in deltas.items():
        pick = rng.choice(ctrl.n_obs, cells_per_pert, replace=cells_per_pert > ctrl.n_obs)
        blocks.append(shifted_counts(ctrl_log[pick], lib[pick], d))
        obs.append(pd.DataFrame({"target": [pert] * cells_per_pert}))
    obs = pd.concat(obs, ignore_index=True)
    obs.index = [f"p{i}" for i in range(len(obs))]
    return ad.AnnData(X=sp.vstack(blocks).tocsr(), obs=obs, var=real.var.copy())


def control_mean_pred(real, perts, cells_per_pert):
    ctrl = real[(real.obs["target"] == "non-targeting").to_numpy()]
    mean = np.rint(np.asarray(ctrl.X.mean(0)).ravel()).astype(np.float32)
    row = sp.csr_matrix(mean)
    blocks = [ctrl.X.copy()] + [sp.vstack([row] * cells_per_pert) for _ in perts]
    obs = pd.DataFrame({"target": ["non-targeting"] * ctrl.n_obs + [p for p in perts for _ in range(cells_per_pert)]})
    obs.index = [f"p{i}" for i in range(len(obs))]
    return ad.AnnData(X=sp.vstack(blocks).tocsr(), obs=obs, var=real.var.copy())


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--real", required=True)
    ap.add_argument("--pseudobulk", required=True)
    ap.add_argument("--grn", default=None, help="grn_ridge .npz for the network method (skip if omitted)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--cells-per-pert", type=int, default=400)
    ap.add_argument("--methods", nargs="+", default=["ctrl_sampled", "ctrl_mean", "mean_effect", "network", "oracle"])
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(a.seed)

    real = ad.read_h5ad(a.real)
    real.obs["target"] = real.obs["target"].astype(str)
    perts = sorted(p for p in real.obs["target"].unique() if p != "non-targeting")
    genes = real.var_names.to_numpy()
    n_genes = len(genes)
    print(f"real: {real.n_obs} cells, {len(perts)} knockdowns, {n_genes} genes")

    pb = np.load(a.pseudobulk, allow_pickle=True)
    assert list(pb["genes"]) == list(genes), "gene order differs between pseudobulk and real file"
    pb_perts = list(pb["perts"])
    delta_all = pb["delta"].astype(np.float64)
    gi = {g: i for i, g in enumerate(genes)}

    deltas = {}
    zero = np.zeros(n_genes)
    deltas["ctrl_sampled"] = {p: zero for p in perts}
    # Leave-one-out mean effect over the *other* knockdowns, self gene set to the median drop.
    tot = delta_all.sum(0)
    me = {}
    for p in perts:
        i = pb_perts.index(p)
        d = (tot - delta_all[i]) / (len(pb_perts) - 1)
        d[gi[p]] = SELF_DROP
        me[p] = d
    deltas["mean_effect"] = me
    deltas["oracle"] = {p: delta_all[pb_perts.index(p)] for p in perts}

    if "network" in a.methods and a.grn:
        g = np.load(a.grn, allow_pickle=True)
        ngenes = list(g["genes"])
        W = g["coef_mean"]
        in_net = [p for p in perts if p in set(ngenes)]
        idx = np.array([ngenes.index(p) for p in in_net])
        D = propagate(W, idx, np.full(len(idx), SELF_DROP), 20)       # network genes x perts
        cols = np.array([gi[x] for x in ngenes])
        net = {}
        for c, p in enumerate(in_net):
            d = np.zeros(n_genes)
            d[cols] = D[:, c]
            net[p] = d
        for p in perts:
            if p not in net:  # not in the network: fall back to self-drop only
                d = np.zeros(n_genes); d[gi[p]] = SELF_DROP; net[p] = d
        deltas["network"] = net
        del W, g

    from cell_eval import MetricsEvaluator

    rows = []
    for m in a.methods:
        if m == "ctrl_mean":
            pred = control_mean_pred(real, perts, a.cells_per_pert)
        elif m in deltas:
            pred = build_pred(real, deltas[m], a.cells_per_pert, np.random.default_rng(a.seed))
        else:
            print(f"skipping {m}"); continue
        print(f"== {m}: pred {pred.n_obs} cells", flush=True)
        ev = MetricsEvaluator(adata_pred=pred, adata_real=real.copy(), control_pert="non-targeting",
                              pert_col="target", outdir=str(out / m), num_threads=4)
        res, agg = ev.compute(profile="vcc", write_csv=True)
        res = res.to_pandas()
        res["method"] = m
        rows.append(res)
        print(res.drop(columns=["method"]).describe().loc[["mean", "50%"]].round(4).to_string(), flush=True)

    df = pd.concat(rows, ignore_index=True)
    df.to_csv(out / "per_pert_metrics.csv", index=False)
    summ = df.groupby("method").mean(numeric_only=True)
    summ.to_csv(out / "summary.csv")
    print("\n" + summ.round(4).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile h1_pseudobulk.py
"""Batch-matched pseudobulk knockdown effects for the 2025 H1 training set.

Streams the 15 GB CSR .h5ad in row chunks, normalizes each cell to log1p(CP10k), and
accumulates per-(target_gene, batch) sums. The true effect of knocking down gene k is

    delta_k = mean over k's cells of (x_cell - control mean of the cell's own batch)

so batch differences don't leak into the effect. Writes an .npz with delta (perts x
genes), control mean, gene names, perturbation names and cell counts.

    python h1_pseudobulk.py adata_Training.h5ad h1_pseudobulk.npz
"""

import sys
from pathlib import Path

import h5py
import numpy as np
import scipy.sparse as sp

CHUNK = 4000


def _col(f, name):
    g = f["obs"][name]
    cats = np.array([c.decode() if isinstance(c, bytes) else c for c in g["categories"][:]], dtype=object)
    return cats[g["codes"][:]]


def main(src, dst):
    with h5py.File(src, "r") as f:
        n, n_genes = (int(v) for v in f["X"].attrs["shape"])
        var = f["var"]["_index"][:]
        genes = np.array([g.decode() if isinstance(g, bytes) else g for g in var], dtype=object)
        target, batch = _col(f, "target_gene"), _col(f, "batch")
        perts, p_idx = np.unique(target, return_inverse=True)
        batches, b_idx = np.unique(batch, return_inverse=True)
        key = p_idx * len(batches) + b_idx
        sums = np.zeros((len(perts) * len(batches), n_genes), dtype=np.float64)
        counts = np.bincount(key, minlength=sums.shape[0])

        indptr = f["X/indptr"][:]
        data_ds, idx_ds = f["X/data"], f["X/indices"]
        LOG1P_COUNTS = np.modf(data_ds[:100000])[0].max() > 1e-3
        print("input is log1p(counts):", LOG1P_COUNTS)
        for start in range(0, n, CHUNK):
            stop = min(start + CHUNK, n)
            lo, hi = indptr[start], indptr[stop]
            data = data_ds[lo:hi].astype(np.float64)
            if LOG1P_COUNTS:  # the 2025 support-set competition_train.h5 stores log1p(counts)
                data = np.rint(np.expm1(data))
            cols = idx_ds[lo:hi]
            row_ptr = indptr[start:stop + 1] - lo
            rows = np.repeat(np.arange(stop - start), np.diff(row_ptr))
            lib = np.bincount(rows, weights=data, minlength=stop - start)
            vals = np.log1p(data * (1e4 / lib[rows]))
            X = sp.csr_matrix((vals, cols, row_ptr), shape=(stop - start, n_genes))
            # Indicator (groups x cells) @ cells sums each cell's row into its (pert, batch) group.
            k = key[start:stop]
            ind = sp.csr_matrix((np.ones(len(k)), (k, np.arange(len(k)))), shape=(sums.shape[0], len(k)))
            g = np.unique(k)
            sums[g] += (ind[g] @ X).toarray()
            if (start // CHUNK) % 10 == 0:
                print(f"{stop}/{n}", flush=True)

    sums = sums.reshape(len(perts), len(batches), n_genes)
    counts = counts.reshape(len(perts), len(batches))
    ctrl = int(np.flatnonzero(perts == "non-targeting")[0])
    ctrl_batch_mean = sums[ctrl] / np.maximum(counts[ctrl], 1)[:, None]           # batches x genes
    ctrl_mean = sums[ctrl].sum(0) / counts[ctrl].sum()
    # Each perturbed cell is compared with its own batch's controls.
    pert_sum = sums.sum(1)
    matched_ctrl = counts @ ctrl_batch_mean                                        # perts x genes
    n_cells = counts.sum(1)
    delta = (pert_sum - matched_ctrl) / n_cells[:, None]
    keep = np.arange(len(perts)) != ctrl
    np.savez_compressed(dst, delta=delta[keep].astype(np.float32), ctrl_mean=ctrl_mean.astype(np.float32),
                        genes=genes.astype(str), perts=perts[keep].astype(str), n_cells=n_cells[keep])
    print(f"{keep.sum()} perturbations x {n_genes} genes -> {dst}")


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
%%writefile h1_extract_perts.py
"""Pull a test set of real H1 knockdown cells (raw counts) for scoring with cell-eval.

Picks `--n-perts` knockdowns stratified by effect size (strong / medium / weak, from the
pseudobulk file), keeps up to `--cells-per-pert` cells for each, plus `--n-controls`
non-targeting cells. Writes an .h5ad with obs column `target` (cell-eval's default).

    python h1_extract_perts.py adata_Training.h5ad h1_pseudobulk.npz h1_real_test.h5ad
"""

import argparse

import anndata as ad
import numpy as np


def main():
    p = argparse.ArgumentParser()
    p.add_argument("src")
    p.add_argument("pseudobulk")
    p.add_argument("dst")
    p.add_argument("--n-perts", type=int, default=40)
    p.add_argument("--cells-per-pert", type=int, default=400)
    p.add_argument("--n-controls", type=int, default=2000)
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args()

    pb = np.load(a.pseudobulk, allow_pickle=True)
    perts, delta = pb["perts"], pb["delta"]
    eff = np.abs(delta).mean(1)
    order = np.argsort(-eff)
    n = a.n_perts
    pick = np.concatenate([order[: n * 3 // 8], order[len(order) // 2 - n * 3 // 16: len(order) // 2 + n * 3 // 16],
                           order[-(n - n * 3 // 8 - 2 * (n * 3 // 16)):]])
    chosen = sorted(set(perts[pick]))
    print(f"{len(chosen)} knockdowns:", ", ".join(chosen))

    rng = np.random.default_rng(a.seed)
    full = ad.read_h5ad(a.src, backed="r")
    tg = full.obs["target_gene"].astype(str).to_numpy()
    keep = []
    for g in chosen + ["non-targeting"]:
        idx = np.flatnonzero(tg == g)
        cap = a.n_controls if g == "non-targeting" else a.cells_per_pert
        keep.extend(rng.choice(idx, min(cap, len(idx)), replace=False))
    keep = np.sort(np.array(keep))
    sub = full[keep].to_memory()
    sub.obs["target"] = sub.obs["target_gene"].astype(str)
    sub.write_h5ad(a.dst, compression="gzip")
    print(f"{sub.n_obs} cells x {sub.n_vars} genes -> {a.dst}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile screen_transfer_h1.py
"""Borrow known H1 knockdown effects, weighted by gene similarity learned from a public screen.

For each H1 knockdown k (leave-one-out over the 150), predict its change as a weighted average
of the other knockdowns' real H1 changes, where the weights come from how alike k and j look in
a public Perturb-seq screen (Replogle 2022 K562 genome-wide and RPE1, Nadig 2025 HepG2 and Jurkat; z-scored pseudobulk):

    pred_k = sum_j w_kj * delta_j / sum_j w_kj,   w_kj = sim(k, j)^+ over the m most similar j

Also tested:
  shrink     pred = (1-a) * mean_effect + a * similarity-weighted average, a in {0.5, 0.75}
  direct     mean_effect + beta * (k's own screen profile mapped onto H1 genes), beta fit on the
             other knockdowns (leave-one-out), so no H1 answer for k leaks in
  uniform    every other knockdown weighted equally (= the 0.19 mean-effect baseline)
  oracle     similarity from k's true H1 effect (the 0.46 ceiling)

Knockdowns absent from a screen fall back to uniform. Scored with eval_h1.score over the same
10,993 genes as the earlier RESULTS.md runs (target gene excluded).

    python screen_transfer_h1.py --pseudobulk h1_pseudobulk.npz --genes-from grn_h1_pc0.top_edges.csv \
        --screen k562=K562_gwps_normalized_bulk_01.h5ad rpe1=rpe1_normalized_bulk_01.h5ad \
        hepg2=hepg2_bulk.h5ad jurkat=jurkat_bulk.h5ad --out out_dir/
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))
from eval_h1 import score  # noqa: E402


def corr_rows(M):
    Z = M - M.mean(1, keepdims=True)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    return Z @ Z.T


def load_screen(path):
    """Return (profiles: perturbed gene -> z-profile, readout gene names). One row per target gene."""
    import anndata as ad
    a = ad.read_h5ad(path)
    X = np.asarray(a.X, dtype=np.float64)
    X[~np.isfinite(X)] = 0                              # z-scores are inf where a readout gene has zero control variance
    np.clip(X, -10, 10, out=X)
    tgt = a.obs.index.str.split("_").str[1].to_numpy()
    df = pd.DataFrame(X).groupby(tgt).mean()          # a few genes have 2 transcripts: average them
    return df, a.var["gene_name"].to_numpy()


def screen_similarity(df, sgenes, perts, exclude_targets=True):
    """Correlation of screen profiles between H1 knockdowns. NaN where either is missing."""
    have = [p for p in perts if p in df.index]
    M = df.loc[have].to_numpy().copy()
    if exclude_targets:                                 # don't let the knocked-down genes' own drops drive similarity
        M = M[:, ~np.isin(sgenes, have)]
    S_have = corr_rows(M)
    S = np.full((len(perts), len(perts)), np.nan)
    idx = np.array([perts.index(p) for p in have])
    S[np.ix_(idx, idx)] = S_have
    return S


def predict_from_sim(S, T, i, m, signed=False):
    s = S[i].copy(); s[i] = np.nan
    ok = np.flatnonzero(np.isfinite(s))
    if len(ok) == 0 or m == 0:
        return None
    key = np.abs(s[ok]) if signed else s[ok]
    order = ok[np.argsort(-key)[:m]]
    w = s[order] if signed else np.clip(s[order], 0, None)
    if np.abs(w).sum() == 0:
        return None
    return (w[:, None] * T[order]).sum(0) / np.abs(w).sum()


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--pseudobulk", required=True)
    ap.add_argument("--genes-from", required=True, help="CSV whose 'target' column lists the genes to score on")
    ap.add_argument("--screen", nargs="+", required=True, metavar="NAME=PATH",
                    help="one or more screens, e.g. k562=K562_gwps_normalized_bulk_01.h5ad rpe1=...")
    ap.add_argument("--out", required=True)
    ap.add_argument("--top", nargs="+", type=int, default=[5, 10, 20, 40])
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    pb = np.load(a.pseudobulk, allow_pickle=True)
    delta_all, pb_genes, perts = pb["delta"].astype(np.float64), list(pb["genes"]), list(pb["perts"])
    genes = list(dict.fromkeys(pd.read_csv(a.genes_from, usecols=["target"])["target"]))
    gi_pb = {g: i for i, g in enumerate(pb_genes)}
    cols = np.array([gi_pb[g] for g in genes])
    gi = {g: i for i, g in enumerate(genes)}
    keep = [p for p in perts if p in gi]
    P = np.array([perts.index(p) for p in keep])
    T = delta_all[P][:, cols]                     # knockdowns x genes, real H1 changes
    K = np.array([gi[p] for p in keep])
    n = len(keep)
    print(f"{n} H1 knockdowns scored over {len(genes)} genes")

    screens = {nm: load_screen(pth) for nm, pth in (x.split("=", 1) for x in a.screen)}
    sims = {}
    for name, (df, sgenes) in screens.items():
        sims[name] = screen_similarity(df, sgenes, keep)
        print(f"{name}: {np.isfinite(np.diag(sims[name])).sum()} of {n} knockdowns in screen")
    if len(sims) > 1:    # average over the screens that have both genes, else whichever has them
        sims["+".join(sims)] = np.nanmean(np.stack(list(sims.values())), 0)
        if {"k562", "rpe1"} <= set(sims):
            sims["k562+rpe1"] = np.nanmean(np.stack([sims["k562"], sims["rpe1"]]), 0)
    sims["oracle"] = corr_rows(T)

    mean_loo = (T.sum(0)[None, :] - T) / (n - 1)
    rows = []

    def add(method, m, i, pred, covered):
        rows.append({"method": method, "top": m, "pert": keep[i], "in_screen": covered, **score(pred, T[i], K[i])})

    for i in range(n):
        add("uniform", n - 1, i, mean_loo[i], True)

    for name, S in sims.items():
        for m in a.top:
            for signed in (False, True):
                if signed and name == "oracle":
                    continue
                tag = name + ("_signed" if signed else "")
                for i in range(n):
                    pred = predict_from_sim(S, T, i, m, signed)
                    covered = pred is not None
                    if pred is None:
                        pred = mean_loo[i]
                    add(tag, m, i, pred, covered)
                    if not signed and name != "oracle":
                        for alpha in (0.5, 0.75):
                            add(f"{name}_shrink{alpha}", m, i, (1 - alpha) * mean_loo[i] + alpha * pred, covered)

    # Direct: k's own screen profile, mapped onto H1 genes, scaled by beta fit on the other knockdowns.
    for name, (df, sgenes) in screens.items():
        sg = {g: c for c, g in enumerate(sgenes)}
        shared = np.array([gi[g] for g in sgenes if g in gi])
        scol = np.array([sg[g] for g in sgenes if g in gi])
        prof = np.zeros((n, len(genes))); has = np.zeros(n, bool)
        for i, p in enumerate(keep):
            if p in df.index:
                prof[i, shared] = df.loc[p].to_numpy()[scol]; prof[i, K[i]] = 0; has[i] = True
        R = T - mean_loo                                  # residual after the shared response
        num = (prof * R).sum(1); den = (prof * prof).sum(1)
        for i in range(n):
            o = has & (np.arange(n) != i)
            beta = num[o].sum() / den[o].sum()
            add(f"direct_{name}", 0, i, mean_loo[i] + beta * prof[i], bool(has[i]))
        print(f"direct_{name}: beta ~ {num[has].sum() / den[has].sum():.4f}")

    df = pd.DataFrame(rows)
    df.to_csv(out / "screen_transfer.csv", index=False)
    cols_ = ["pearson_all", "pearson_top100", "sign_top50"]
    summ = df.groupby(["method", "top"])[cols_].mean()
    cov = df[df.in_screen].groupby(["method", "top"])["pearson_all"].agg(["mean", "size"])
    summ["pearson_all_in_screen"] = cov["mean"]; summ["n_in_screen"] = cov["size"]
    summ = summ.sort_values("pearson_all", ascending=False)
    summ.to_csv(out / "screen_transfer_summary.csv")
    print(summ.round(3).to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile screen_bulk_from_cells.py
"""Turn a single-cell Perturb-seq .h5/.h5ad (log-normalized X, obs target_gene) into a z-scored
pseudobulk file in the same layout as Replogle's *_normalized_bulk_01.h5ad, so
screen_transfer_h1.load_screen can read it.

Profile of knockdown k = (mean of k's cells - mean of controls in the same batch) / control std,
averaged over batches weighted by k's cells. Readout genes with < min_detect controls expressing
them are dropped.

    python screen_bulk_from_cells.py hepg2.h5 hepg2_bulk.h5ad [--batch batch_var]
"""

import argparse

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("src"); ap.add_argument("dst")
    ap.add_argument("--batch", default="batch_var")
    ap.add_argument("--control", default="non-targeting")
    ap.add_argument("--min-detect", type=float, default=0.05)
    a = ap.parse_args()

    x = ad.read_h5ad(a.src)
    X = x.X.toarray() if sp.issparse(x.X) else np.asarray(x.X)
    X = X.astype(np.float32)
    tg = x.obs["target_gene"].astype(str).to_numpy()
    bt = x.obs[a.batch].astype(str).to_numpy() if a.batch in x.obs else np.zeros(len(tg), str)
    ctrl = tg == a.control
    keep_g = (X[ctrl] > 0).mean(0) >= a.min_detect
    X = X[:, keep_g]; genes = x.var_names.to_numpy()[keep_g]
    sd = X[ctrl].std(0) + 1e-3
    gm = X[ctrl].mean(0)
    bmean = {b: X[ctrl & (bt == b)].mean(0) if (ctrl & (bt == b)).sum() >= 20 else gm for b in np.unique(bt)}
    rows, names = [], []
    for k in sorted(set(tg) - {a.control}):
        idx = np.flatnonzero(tg == k)
        ref = np.stack([bmean[b] for b in bt[idx]]).mean(0)
        rows.append((X[idx].mean(0) - ref) / sd)
        names.append(f"{len(names)}_{k}_bulk")
    out = ad.AnnData(X=np.stack(rows).astype(np.float32), obs=pd.DataFrame(index=names),
                     var=pd.DataFrame({"gene_name": genes}, index=genes))
    out.write_h5ad(a.dst)
    print(f"{out.n_obs} knockdowns x {out.n_vars} readout genes -> {a.dst}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile celleval_screen_h1.py
"""cell-eval on real H1 knockdowns for screen-similarity effect borrowing, next to the baselines.

Builds each submission the same way as logo-shap/v2/celleval_h1.py (resample 400 control
cells per knockdown and shift them by the predicted change), with these methods:

  ctrl_sampled   no change (chance)
  mean_effect    leave-one-out mean of the other 149 H1 knockdowns (the 0.19 Pearson baseline)
  screen_sim     similarity-weighted mean of the other knockdowns, similarity = correlation of
                 their profiles in public screens (default: Replogle K562 GWPS + RPE1), top-m,
                 shrunk halfway to mean_effect; falls back to mean_effect if k is in no screen
  oracle         the real change

The knocked-down gene itself is set to the H1 median drop in every non-oracle method.

    python celleval_screen_h1.py --real h1_real_test.h5ad --pseudobulk h1_pseudobulk.npz \
        --screens K562_gwps_normalized_bulk_01.h5ad rpe1_normalized_bulk_01.h5ad --out out/
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

HERE = Path(__file__).parent
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(Path(__file__).parent))
from celleval_h1 import SELF_DROP, build_pred  # noqa: E402
from screen_transfer_h1 import load_screen, predict_from_sim, screen_similarity  # noqa: E402


def screen_sim_deltas(delta_all, perts, screens, m, alpha):
    sims = [screen_similarity(df, sg, perts) for df, sg in screens]
    S = np.nanmean(np.stack(sims), 0) if len(sims) > 1 else sims[0]
    n = len(perts)
    mean_loo = (delta_all.sum(0)[None, :] - delta_all) / (n - 1)
    out, covered = {}, 0
    for i in range(n):
        p = predict_from_sim(S, delta_all, i, m)
        if p is None:
            out[perts[i]] = mean_loo[i]
        else:
            out[perts[i]] = (1 - alpha) * mean_loo[i] + alpha * p
            covered += 1
    print(f"screen_sim: {covered}/{n} knockdowns covered by a screen")
    return out, mean_loo


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--real", required=True)
    ap.add_argument("--pseudobulk", required=True)
    ap.add_argument("--screens", nargs="+", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--top", type=int, default=10)
    ap.add_argument("--alpha", type=float, default=0.5)
    ap.add_argument("--cells-per-pert", type=int, default=400)
    ap.add_argument("--methods", nargs="+", default=["ctrl_sampled", "mean_effect", "screen_sim", "oracle"])
    ap.add_argument("--threads", type=int, default=4)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    real = ad.read_h5ad(a.real)
    real.obs["target"] = real.obs["target"].astype(str)
    test = sorted(p for p in real.obs["target"].unique() if p != "non-targeting")
    genes = real.var_names.to_numpy()
    gi = {g: i for i, g in enumerate(genes)}
    print(f"real: {real.n_obs} cells, {len(test)} knockdowns, {len(genes)} genes")

    pb = np.load(a.pseudobulk, allow_pickle=True)
    assert list(pb["genes"]) == list(genes), "gene order differs between pseudobulk and real file"
    perts = [str(p) for p in pb["perts"]]
    delta_all = pb["delta"].astype(np.float64)

    screens = [load_screen(s) for s in a.screens]
    sim, mean_loo = screen_sim_deltas(delta_all, perts, screens, a.top, a.alpha)

    def with_self(d, p):
        d = d.copy(); d[gi[p]] = SELF_DROP; return d

    deltas = {
        "ctrl_sampled": {p: np.zeros(len(genes)) for p in test},
        "mean_effect": {p: with_self(mean_loo[perts.index(p)], p) for p in test},
        "screen_sim": {p: with_self(sim[p], p) for p in test},
        "oracle": {p: delta_all[perts.index(p)] for p in test},
    }

    from cell_eval import MetricsEvaluator
    rows = []
    for m in a.methods:
        pred = build_pred(real, deltas[m], a.cells_per_pert, np.random.default_rng(a.seed))
        print(f"== {m}: pred {pred.n_obs} cells", flush=True)
        ev = MetricsEvaluator(adata_pred=pred, adata_real=real.copy(), control_pert="non-targeting",
                              pert_col="target", outdir=str(out / m), num_threads=a.threads)
        res, _ = ev.compute(profile="vcc", write_csv=True)
        res = res.to_pandas(); res["method"] = m
        rows.append(res)
        print(res.drop(columns=["method"]).mean(numeric_only=True).round(4).to_string(), flush=True)

    df = pd.concat(rows, ignore_index=True)
    df.to_csv(out / "per_pert_metrics.csv", index=False)
    summ = df.groupby("method").mean(numeric_only=True)
    summ.to_csv(out / "summary.csv")
    print("\n" + summ.round(4).to_string())


if __name__ == "__main__":
    main()


## Download data
Replogle pseudobulk files from figshare, and the 2025 VCC support set (H1 training cells plus Nadig HepG2/Jurkat subsets) from Arc's public bucket.

In [ ]:
%%bash
set -e
cd /content/data
curl -sSL -o K562_gwps_normalized_bulk_01.h5ad https://ndownloader.figshare.com/files/35773217
curl -sSL -o rpe1_normalized_bulk_01.h5ad https://ndownloader.figshare.com/files/35775512
curl -sSL -o css.zip https://storage.googleapis.com/vcc_data_prod/datasets/state/competition_support_set.zip
unzip -o -j css.zip competition_support_set/competition_train.h5 competition_support_set/hepg2.h5 competition_support_set/jurkat.h5
rm css.zip
ls -la

## Build inputs
H1 batch-matched pseudobulk effects (150 knockdowns), the 40-knockdown cell-eval test set, and z-scored pseudobulk profiles for HepG2 and Jurkat.

In [ ]:
!python h1_pseudobulk.py /content/data/competition_train.h5 /content/data/h1_pseudobulk.npz
!python h1_extract_perts.py /content/data/competition_train.h5 /content/data/h1_pseudobulk.npz /content/data/h1_real_test.h5ad
!python screen_bulk_from_cells.py /content/data/hepg2.h5 /content/data/hepg2_bulk.h5ad
!python screen_bulk_from_cells.py /content/data/jurkat.h5 /content/data/jurkat_bulk.h5ad

## Score genes
The earlier runs scored on the 10,993 genes of the H1 control-cell network, so this writes that exact list to keep the Pearson numbers comparable.

In [ ]:
%%writefile /content/data/score_genes.csv
target
NOC2L
KLHL17
ISG15
AGRN
C1orf159
SDF4
UBE2J2
ACAP3
PUSL1
INTS11
CPTP
DVL1
MXRA8
AURKAIP1
CCNL2
VWA1
ATAD3B
ATAD3A
TMEM240
SSU72
FNDC10
MIB2
MMP23B
CDK11B
CDK11A
NADK
GNB1
TMEM52
PRKCZ
FAAP20
SKI
RER1
PEX10
PANK4
PRXL2B
ARHGEF16
MEGF6
TPRG1L
WRAP73
LRRC47
CEP104
C1orf174
AJAP1
NPHP4
KCNAB2
ICMT
ACOT7
TNFRSF25
PLEKHG5
NOL9
ZBTB48
KLHL21
PHF13
THAP3
DNAJC11
CAMTA1
VAMP3
PER3
ERRFI1
RERE
ENO1
SPSB1
SLC25A33
TMEM201
PIK3CD
CLSTN1
CTNNBIP1
LZIC
NMNAT1
RBP7
UBE4B
KIF1B
PGD
CENPS
DFFA
PEX14
CASZ1
SRM
EXOSC10
MTOR
UBIAD1
DISP3
FBXO2
FBXO44
MAD2L2
DRAXIN
AGTRAP
MTHFR
CLCN6
KIAA2013
PLOD1
MFN2
MIIP
TNFRSF8
VPS13D
DHRS3
PDPN
PRDM2
KAZN
TMEM51
EFHD2
CASP9
DNAJC16
AGMAT
DDI2
PLEKHM2
FBLIM1
SPEN
ZBTB17
EPHA2
ARHGEF19
FBXO42
SZRD1
NECAP2
CROCC
MFAP2
ATP13A2
SDHB
PADI2
RCC2
ARHGEF10L
ACTL8
IGSF21
KLHDC7A
ALDH4A1
IFFO2
UBR4
EMC1
MRTO4
AKR7A2
SLC66A1
CAPZB
MICOS10
NBL1
TMCO4
OTUD3
UBXN10
CAMK2N1
MUL1
CDA
PINK1
DDOST
KIF17
SH2D5
HP1BP3
EIF4G3
ECE1
ALPL
RAP1GAP
USP48
HSPG2
ZBTB40
EPHA8
EPHB2
KDM1A
LUZP1
HTR1D
HNRNPR
ZNF436
ASAP3
E2F2
ID3
ELOA
PITHD1
GALE
HMGCL
FUCA1
PNRC2
SRSF10
STPG1
NIPAL3
RCAN3
SRRM1
CLIC4
SYF2
RSRP1
TMEM50A
MACO1
LDLRAP1
MAN1C1
SELENON
MTFR1L
AUNIP
PAQR7
STMN1
PAFAH2
PDIK1L
ZNF593
CNKSR1
CEP85
SH3BGRL3
UBXN11
CD52
CRYBG2
LIN28A
DHDDS
ARID1A
PIGV
ZDHHC18
GPN2
GPATCH3
NUDC
KDF1
TRNP1
TENT5B
SLC9A1
WDTC1
TMEM222
SYTL1
MAP3K6
GPR3
WASF2
AHDC1
IFI6
FAM76A
STX12
PPP1R8
THEMIS2
RPA2
SMPDL3B
XKR8
EYA3
DNAJC8
ATP5IF1
SESN2
MED18
PHACTR4
RCC1
TRNAU1AP
TAF12
RAB42
GMEB1
YTHDF2
OPRD1
EPB41
SRSF4
MECR
PTPRU
SDC3
PUM1
NKAIN1
SNRNP40
ZCCHC17
FABP3
SERINC2
TINAGL1
PEF1
COL16A1
ADGRB2
KHDRBS1
TMEM39B
KPNA6
TXLNA
CCDC28B
IQCC
TMEM234
EIF3I
LCK
HDAC1
MARCKSL1
BSDC1
ZBTB8B
ZBTB8A
RBBP4
KIAA1522
YARS
S100PBP
TMEM54
RNF19B
ZNF362
PHC2
CSMD2
C1orf94
SMIM12
DLGAP3
TMEM35B
ZMYM6
ZMYM1
SFPQ
ZMYM4
KIAA0319L
NCDN
PSMB2
C1orf216
CLSPN
AGO4
AGO1
AGO3
ADPRHL2
COL8A2
TRAPPC3
MAP7D1
THRAP3
SH3D21
EVA1B
STK40
LSM10
GRIK3
ZC3H12A
MEAF6
SNIP1
DNALI1
GNL2
C1orf109
CDCA8
MANEAL
YRDC
C1orf122
MTF1
INPP5B
SF3A3
FHL3
UTP11
POU3F1
RRAGC
MYCBP
RHBDL2
AKIRIN1
NDUFS5
MACF1
PABPC4
HEYL
PPIE
TRIT1
MYCL
MFSD2A
PPT1
RLF
ZMPSTE24
SMAP2
ZFP69B
ZFP69
EXO5
ZNF684
RIMS3
NFYC
KCNQ4
CITED4
CTPS1
SCMH1
FOXO6
FOXJ3
RIMKLA
PPCS
PPIH
YBX1
P3H1
SVBP
ERMAP
ZNF691
SLC2A1
EBNA1BP2
TMEM125
C1orf210
CDC20
ELOVL1
MED8
HYI
PTPRF
KDM4A
ST3GAL3
IPO13
DPH2
ATP6V0B
B4GALT2
CCDC24
SLC6A9
DMAP1
ERI3
RNF220
KIF2C
PLK3
EIF2B3
HECTD3
UROD
ZSWIM5
HPDL
MUTYH
TOE1
TESK2
MMACHC
PRDX1
AKR1A1
NASP
GPBP1L1
TMEM69
MAST2
PIK3R3
POMGNT1
LURAP1
RAD54L
LRRC41
UQCRH
NSUN4
MKNK1
ATPAF1
EFCAB14
STIL
CMPK1
SPATA6
BEND5
FAF1
RNF11
TTC39A
EPS15
OSBPL9
NRDC
RAB3B
TXNDC12
KTI12
BTF3L4
ZFYVE9
CC2D1B
ORC1
PRPF38A
TUT4
GPX7
COA7
ZYG11B
ZYG11A
SCP2
CPT2
CZIB
MAGOH
LRP8
NDC1
YIPF1
HSPB11
LRRC42
TMEM59
TCEANC2
CYB5RL
SSBP3
ACOT11
TTC4
PARS2
DHCR24
PCSK9
USP24
PLPP3
PRKAA2
DAB1
OMA1
MYSM1
JUN
FGGY
HOOK1
TM2D1
PATJ
L1TD1
USP1
DOCK7
ATG4C
FOXD3
ALG6
ITGB3BP
EFCAB7
PGM1
ROR1
CACHD1
RAVER2
JAK1
AK4
DNAJC6
LEPROT
LEPR
PDE4B
MIER1
SLC35D1
SERBP1
GADD45A
GNG12
DEPDC1
LRRC40
SRSF11
ANKRD13C
HHLA3
CTH
ZRANB2
FPGT
CRYZ
TYW3
SLC44A5
ACADM
RABGGTB
ST6GALNAC3
ST6GALNAC5
PIGK
AK5
AC118549.1
USP33
MIGA1
NEXN
FUBP1
DNAJB4
ADGRL2
TTLL7
PRKACB
SAMD13
RPF1
GNG5
CTBS
SSX2IP
LPAR3
MCOLN2
MCOLN3
SYDE2
C1orf52
BCL10
DDAH1
CCN1
ZNHIT6
ODF2L
SH3GLB1
SELENOF
HS2ST1
LMO4
PKN2
GTF2B
KYAT3
RBMXL1
LRRC8B
LRRC8C
LRRC8D
ZNF326
ZNF644
HFM1
CDC7
TGFBR3
BTBD8
GLMN
RPAP2
EVI5
DIPK1A
MTF2
TMED5
CCDC18
DR1
FNBP1L
BCAR3
DNTTIP2
GCLM
ARHGAP29
ABCD3
F3
SLC44A3
CNN3
ALG14
TLCD4
RWDD3
PTBP2
SNX7
PLPPR5
PLPPR4
FRRS1
AGL
SLC35A3
MFSD14A
SASS6
TRMT13
DBT
RTCA
CDC14A
EXTL2
SLC30A7
DPH5
OLFM3
COL11A1
RNPC3
PRMT6
VAV3
SLC25A24
FAM102B
HENMT1
PRPF38B
STXBP3
GPSM2
CLCC1
WDR47
TAF13
TMEM167B
SARS
CELSR2
PSRC1
SORT1
PSMA5
GNAI3
AMPD2
GSTM3
CSF1
AHCYL1
STRIP1
KCNC4
RBM15
LAMTOR5
LRIF1
DRAM2
CEPT1
PIFO
WDR77
ATP5PB
RAP1A
INKA2
DDX20
CTTNBP2NL
ST7L
CAPZA1
MOV10
RHOC
PPM1J
SLC16A1
LRIG2
MAGI3
PHTF1
RSBN1
AP4B1
DCLRE1B
HIPK1
OLFML3
SYT6
TRIM33
BCAS2
DENND2C
NRAS
CSDE1
SIKE1
TSPAN2
VANGL1
SLC22A15
ATP1A1
CD58
IGSF3
PTGFRN
TTF2
TRIM45
MAN1A2
GDAP2
WDR3
WARS2
ZNF697
PHGDH
NOTCH2
SEC22B
RNF115
POLR3C
NUDT17
PIAS3
ANKRD35
PEX11B
LIX1L
POLR3GL
TXNIP
PRKAB2
BCL9
ACP6
GPR89B
HIST2H2BE
SV2A
SF3B4
OTUD7B
VPS45
PLEKHO1
ANP32E
CA14
APH1A
C1orf54
CIART
PRPF3
RPRD2
TARS2
ADAMTSL4
MCL1
ENSA
GOLPH3L
CTSK
ARNT
SETDB1
CERS2
PRUNE1
C1orf56
CDC42SE1
MLLT11
GABPB2
SEMA6C
SCNM1
VPS72
PIP5K1A
PSMD4
ZNF687
PI4KB
RFX5
SELENBP1
PSMB4
POGZ
CGN
TUFT1
SNX27
TDRKH
THEM4
S100A10
S100A11
S100A6
S100A4
S100A16
S100A13
CHTOP
SNAPIN
ILF2
NPR1
INTS3
SLC27A3
GATAD2B
DENND4B
CRTC2
SLC39A1
CREB3L4
JTB
RAB13
C1orf43
UBAP2L
HAX1
ATP8B2
IL6R
UBE2Q1
CHRNB2
ADAR
PMVK
PBXIP1
PYGO2
SHC1
CKS1B
FLAD1
ZBTB7B
ADAM15
EFNA3
EFNA1
SLC50A1
DPM3
KRTCAP2
TRIM46
MUC1
MTX1
GBA
FAM189B
SCAMP3
FDPS
RUSC1
ASH1L
MSTO1
YY1AP1
DAP3
GON4L
SYT11
RIT1
KHDC4
ARHGEF2
SSR2
UBQLN4
LAMTOR2
RAB25
MEX3A
LMNA
SEMA4A
SLC25A44
PMF1
BGLAP
PAQR6
SMG5
GLMP
CCT3
TSACC
MEF2D
IQGAP3
NAXE
GPATCH4
NES
CRABP2
ISG20L2
RRNAD1
HDGF
PRCC
SH2D2A
ARHGEF11
ETV3
KIRREL1
AIM2
CADM3
DUSP23
TAGLN2
IGSF9
PIGM
IGSF8
ATP1A2
PEA15
DCAF8
PEX19
COPA
NCSTN
VANGL2
ITLN2
F11R
TSTD1
USF1
KLHDC9
PFDN2
NIT1
DEDD
UFC1
USP21
PPOX
B4GALT3
ADAMTS4
NDUFS2
TOMM40L
MPZ
DUSP12
ATF6
OLFML2B
NOS1AP
C1orf226
UHMK1
UAP1
RGS5
NUF2
PBX1
MGST3
ALDH9A1
TMCO1
UCK2
FAM78B
POGK
TADA1
POU2F1
CREG1
MPZL1
MPC2
DCAF6
GPR161
TIPRL
SFT2D2
ATP1B1
NME7
BLZF1
CCDC181
SLC19A2
C1orf112
METTL18
SCYL3
GORAB
PRRC2C
VAMP4
EEF1AKNMT
DNM3
PIGC
SUCO
PRDX6
ANKRD45
KLHL20
CENPL
DARS2
ZBTB37
RC3H1
RABGAP1L
KIAA0040
COP1
ASTN1
RASAL2
RALGPS2
FAM20B
TOR3A
ABL2
SOAT1
TOR1AIP2
TOR1AIP1
CEP350
QSOX1
ACBD6
XPR1
STX6
IER5
GLUL
RGS16
NPL
DHX9
LAMC1
LAMC2
NMNAT2
SMG7
ARPC5
TSEN15
C1orf21
EDEM3
NIBAN1
RNF2
TRMT1L
SWT1
IVNS1ABP
HMCN1
TPR
ODR4
PLA2G4A
RGS2
UCHL5
RO60
GLRX2
CDC73
KCNT2
ASPM
ZBTB41
DENND1B
NEK7
NR5A2
ZNF281
KIF14
DDX59
CAMSAP2
INAVA
KIF21B
TMEM9
TNNT2
LAD1
PHLDA3
CSRP1
NAV1
IPO9
SHISA4
TIMM17A
RNPEP
ELF3
ARL8A
UBE2T
PPP1R12B
SYT2
KDM5B
RABIF
KLHL12
ADIPOR1
CYB5R1
TMEM183A
PPFIA4
OPTC
ATP2B4
SNRPE
SOX13
ETNK2
PLEKHA6
PPP1R15B
PIK3C2B
MDM4
LRRN2
NFASC
TMEM81
RBBP5
DSTYK
TMCC2
NUAK2
LEMD1
CDK18
ELK4
SLC45A3
NUCKS1
RAB29
SLC41A1
IKBKE
EIF2D
DYRK3
MAPKAPK2
C1orf116
PFKFB2
YOD1
CD55
CR2
CD46
C1orf74
IRF6
UTP25
SYT14
RCOR3
TRAF5
SLC30A1
NEK2
LPGAT1
INTS7
DTL
PPP2R5A
PACC1
NENF
ATF3
BATF3
NSL1
TATDN3
FLVCR1
VASH2
ANGEL2
SMYD2
PTPN14
CENPF
KCTD3
GPATCH2
RRP15
LYPLAL1
EPRS
BPNT1
IARS2
RAB3GAP2
MARK1
C1orf115
MARC2
MARC1
TAF1A
MIA3
AIDA
BROX
DISP1
SUSD4
CAPN2
TP53BP2
FBXO28
DEGS1
NVL
CNIH4
WDR26
CNIH3
DNAH14
LBR
ENAH
SRP9
EPHX1
TMEM63A
LEFTY1
PYCR2
LEFTY2
SDE2
H3F3A
ACBD3
LIN9
PARP1
ITPKB
PSEN2
COQ8A
CDC42BPA
ZNF678
SNAP47
JMJD4
ARF1
C1orf35
GUK1
IBA57
OBSCN
TRIM11
HIST3H2A
RNF187
RHOU
RAB4A
CCSAP
ACTA1
NUP133
ABCB10
TAF5L
URB2
GALNT2
PGBD5
COG2
C1orf198
TTC13
ARV1
C1orf131
GNPAT
EXOC8
SPRTN
EGLN1
TSNAX
DISC1
SIPA1L2
MAP10
NTPCR
PCNX2
MAP3K21
KCNK1
COA6
TARBP1
TOMM20
RBM34
ARID4B
GGPS1
TBCE
TBCE-1
B3GALNT2
GNG4
NID1
GPR137B
ERO1B
EDARADD
LGALS8
HEATR1
MTR
RYR2
CHRM3
RGS7
FH
OPN3
CHML
EXO1
PLD5
CEP170
SDCCAG8
AKT3
ZBTB18
ADSS
DESI2
COX20
HNRNPU
EFCAB2
SMYD3
TFB2M
CNST
SCCPDH
AHCTF1
ZNF695
ZNF670
ZNF669
ZNF496
SH3BP5L
ZNF672
ZNF692
PGBD2
FAM110C
SH3YL1
ACP1
ALKAL2
TMEM18
SNTG2
PXDN
EIPR1
TRAPPC12
ADI1
RNASEH1
RNF144A
ID2
KIDINS220
MBOAT2
ASAP2
ITGB1BP1
CPSF3
IAH1
ADAM17
YWHAQ
TAF1B
KLF11
RRM2
ODC1
NOL10
PDIA6
SLC66A3
ROCK2
E2F6
GREB1
LPIN1
TRIB2
NBAS
DDX1
MYCNOS
MYCN
FAM49A
VSNL1
SMC6
GEN1
KCNS3
RDH14
TTC32
WDR35
MATN3
LAPTM4A
SDC1
PUM2
RHOB
LDAH
KLHL29
ATAD2B
UBXN2A
WDCP
FKBP1B
SF3B6
TP53I3
ITSN2
NCOA1
PTRHD1
CENPO
ADCY3
DNAJC27
EFR3B
POMC
DNMT3A
DTNB
ASXL2
KIF3C
RAB10
GAREM2
HADHA
HADHB
SELENOI
SLC35F6
CENPA
DPYSL5
MAPRE3
TMEM214
AGBL5
OST4
KHK
CGREF1
PREB
SLC5A6
ATRAID
CAD
TRIM54
MPV17
GTF3C2
EIF2B4
SNX17
ZNF513
PPM1G
NRBP1
KRTCAP3
IFT172
FNDC4
ZNF512
GPN1
SUPT7L
SLC4A1AP
BABAM2
PPP1CB
TRMT61B
WDR43
YPEL5
LBH
LCLAT1
GALNT14
EHD3
DPY30
SPAST
SLC30A6
YIPF4
BIRC6
TTC27
LTBP1
FAM98A
CRIM1
FEZ2
STRN
HEATR5B
GPATCH11
EIF2AK2
CEBPZOS
CEBPZ
NDUFAF7
PRKD3
QPCT
CDC42EP3
RMDN2
ATL2
HNRNPLL
GALM
SRSF7
GEMIN6
DHX57
MORN2
SOS1
MAP4K3
THUMPD2
SLC8A1
PKDCC
EML4
COX7A2L
KCNG3
MTA3
ZFP36L2
THADA
DYNC2LI1
LRPPRC
PPM1B
PREPL
CAMKMT
SRBD1
EPAS1
ATP6V1E2
RHOQ
PIGF
CRIPT
SOCS5
MCFD2
TTC7A
CALM2
EPCAM
MSH2
KCNK12
MSH6
FBXO11
FOXN2
PPP1R21
STON1
NRXN1
ASB3
CHAC2
ERLEC1
PSME4
ACYP2
SPTBN1
RTN4
CLHC1
MTIF2
CCDC88A
CFAP36
PPP4R3B
EFEMP1
VRK2
FANCL
BCL11A
PAPOLG
REL
PEX13
KIAA1841
C2orf74
USP34
XPO1
FAM161A
CCT4
COMMD1
B3GNT2
TMEM17
EHBP1
WDPCP
MDH1
UGP2
VPS54
PELI1
LGALSL
AFTPH
SERTAD2
CEP68
RAB1A
ACTR2
SPRED2
ETAA1
WDR92
PNO1
PPP3R1
CNRIP1
FBXO48
PROKR1
ANTXR1
GFPT1
NFU1
AAK1
ANXA4
GMCL1
SNRNP27
PCBP1
C2orf42
TIA1
PCYOX1
SNRPG
FAM136A
ADD2
TEX261
NAGK
MCEE
MPHOSPH10
PAIP2B
ZNF638
DYSF
EXOC6B
SPR
SFXN5
RAB11FIP5
SMYD5
PRADC1
FBXO41
ALMS1
TPRKB
DUSP11
STAMBP
ACTG2
DGUOK
TET3
BOLA3
MOB1A
MTHFD2
DCTN1
WDR54
RTKN
INO80B
WBP1
MOGS
TTC31
PCGF1
AUP1
HTRA2
SEMA4F
HK2
POLE4
GCFC2
LRRTM4
CTNNA2
SUCLG1
TRABD2A
TMSB10
KCMF1
TCF7L1
TGOLN2
RETSAT
ELMOD3
CAPG
MAT2A
GGCX
VAMP8
RNF181
TMEM150A
USP39
C2orf68
ST3GAL5
POLR1A
IMMT
KDM3A
CHMP3
RNF103
RMND5A
KRCC1
THNSL2
FOXI3
EIF2AK3
RPIA
ZNF514
ZNF2
AC092835.1
FAHD2A
STARD7
TMEM127
CIAO1
SNRNP200
ITPRIPL1
NCAPH
ARID5A
KANSL3
LMAN2L
CNNM4
CNNM3
SEMA4C
FAHD2B
ANKRD36
COX5B
ACTR1B
TMEM131
INPP4A
COA5
UNC50
MGAT4A
TSGA10
C2orf15
LIPT1
MITD1
TXNDC9
EIF5B
REV1
CHST10
PDCL3
TBC1D8
CNOT11
RNF149
MAP4K4
MFSD9
TMEM182
TGFBRAP1
C2orf49
FHL2
NCK2
ECRG4
UXS1
ST6GAL2
SULT1C4
GCC2
CCDC138
SEPTIN10
SOWAHC
NPHP1
MTLN
BUB1
ACOXL
BCL2L11
MERTK
TMEM87B
ZC3H8
ZC3H6
TTL
POLR1B
CHCHD5
SLC20A1
CKAP2L
SLC35F5
ACTR3
DPP10
DDX18
CCDC93
INSIG2
STEAP3
C2orf76
DBI
TMEM37
TMEM177
PTPN4
EPB41L5
TMEM185B
RALB
GLI2
TFCP2L1
CLASP1
NIFK
TSN
GYPC
BIN1
CYP27C1
ERCC3
MAP3K2
IWS1
LIMS2
WDR33
SFT2D3
POLR2D
AMMECR1L
SAP130
UGGT1
HS6ST1
SMPD4
CCDC115
IMP4
PTPN18
ARHGEF4
FAM168B
PLEKHB2
LYPD1
MGAT5
CCNT2
RAB3GAP1
ZRANB3
R3HDM1
UBXN4
MCM6
DARS
HNMT
SPOPL
NXPH2
GTDC1
ACVR2A
ORC4
MBD5
EPC2
KIF5C
LYPD6B
LYPD6
MMADHC
RND3
RBM43
NMI
RIF1
NEB
ARL5A
CACNB4
STAM2
FMNL2
PRPF40A
ARL6IP6
GALNT13
GPD2
ACVR1
CCDC148
PKP4
TANC1
WDSUB1
BAZ2B
MARCH7
RBMS1
TANK
PSMD14
GCA
FIGN
GRB14
CSRNP3
GALNT3
TTC21B
SCN9A
B3GALT1
STK39
CERS6
SPC25
FASTKD1
PPIG
PHOSPHO2
KLHL23
METTL5
UBR3
ERICH2
GORASP2
TLK1
METTL8
DCAF17
CYBRD1
DYNC1I2
SLC25A12
HAT1
METAP1D
ITGA6
PDK1
MAP3K20
CDCA7
SP3
CIR1
SCRN3
WIPF1
CHN1
ATF2
ATP5MC3
LNPK
MTX2
HNRNPA3
NFE2L2
AGPS
TTC30A
RBM45
OSBPL6
PRKRA
FKBP7
PLEKHA3
TTN
CCDC141
SESTD1
CWC22
UBE2E3
ITGA4
ITPRID2
DNAJC10
FRZB
NCKAP1
NUP35
ZC3H15
ITGAV
FAM171B
TFPI
GULP1
COL5A2
WDR75
ASNSD1
OSGEPL1
ORMDL1
PMS1
HIBCH
MFSD6
NEMP2
NAB1
GLS
STAT1
MYO1B
NABP1
SLC39A10
STK17B
HECW2
CCDC150
GTF3C3
PGAP1
SF3B1
COQ10B
HSPD1
HSPE1
MOB4
C2orf69
TYW5
MAIP1
SPATS2L
SGO2
AOX1
BZW1
CLK1
PPIL3
NIF3L1
ORC2
FAM126B
NDUFB3
CFLAR
CASP10
CASP8
TRAK2
STRADB
TMEM237
ALS2
FZD7
NOP58
BMPR2
FAM117B
WDR12
CARF
NBEAL1
CYP20A1
ABI2
RAPH1
PARD3B
NRP2
INO80D
NDUFS1
EEF1B2
ZDBF2
ADAM23
MDH1B
FASTKD2
KLF7
CREB1
METTL21A
CCNYL1
FZD5
PLEKHM3
IDH1
PIKFYVE
MAP2
RPE
KANSL1L
LANCL1
CPS1
IKZF2
SPAG16
BARD1
ATIC
FN1
MREG
PECR
TMEM169
XRCC5
MARCH4
SMARCAL1
IGFBP2
TNS1
ARPC2
AAMP
PNKD
TMBIM1
CTDSP1
USP37
CNOT9
ZNF142
BCS1L
RNF25
STK36
TTLL4
NHEJ1
CNPPD1
RETREG2
ZFAND2B
ABCB6
ATG9A
ANKZF1
STK16
TUBA4A
DNAJB2
PTPRN
DNPEP
SPEG
GMPPA
CHPF
TMEM198
OBSL1
STK11IP
SLC4A3
EPHA4
FARSB
ACSL3
AP1S3
WDFY1
SERPINE2
CUL3
IRS1
RHBDD1
MFF
AGFG1
SLC19A3
PID1
DNER
TRIP12
FBXO36
SLC16A14
SP110
CAB39
ITM2C
PSMD1
ARMC9
B3GNT7
NCL
PDE6D
COPS7B
DIS3L2
ECEL1
EIF4E2
EFHD1
GIGYF2
NGEF
ATG16L1
DGKD
USP40
HJURP
ARL4C
SH3BP4
AGAP1
IQCA1
ACKR3
COPS8
COL6A3
MLPH
RAB17
LRRFIP1
UBE2F
SCLY
ILKAP
HES6
PER2
TRAF3IP1
ASB1
HDAC4
NDUFA10
COPS9
GPC1
RNPEPL1
KIF1A
MTERF4
PASK
PPP1R7
HDLBP
SEPTIN2
FARP2
STK25
BOK
THAP4
ATG4B
DTYMK
ING5
TRNT1
CRBN
SUMF1
LRRN1
SETMAR
ITPR1
BHLHE40
ARL8B
EDEM1
LMCD1
RAD18
SRGAP3
THUMPD3
SETD5
MTMR14
CPNE9
BRPF1
OGG1
CAMK1
TADA3
ARPC4
RPUSD3
JAGN1
IL17RC
CRELD1
EMC3
FANCD2
BRK1
VHL
TATDN2
SEC13
ATG7
VGLL4
TAMM41
SYN2
TIMP4
TSEN2
MKRN2
RAF1
CAND2
IQSEC1
NUP210
HDAC11
FBLN2
CHCHD4
TMEM43
XPC
LSM3
SLC6A6
GRIP2
CCDC174
FGD5
NR2C2
CAPN7
SH3BP5
METTL6
EAF1
HACL1
BTD
ANKRD28
DPH3
OXNAD1
RFTN1
PLCL2
TBC1D5
SATB1
KCNH8
RAB5A
SGO1
ZNF385D
UBE2E2
UBE2E1
NKIRAS1
NR1D2
TOP2B
NGLY1
LRRC3B
SLC4A7
CMC1
AZI2
TGFBR2
STT3B
OSBPL10
ZNF860
GPD1L
CMTM8
CMTM7
CMTM6
DYNC1LI1
CNOT10
TRIM71
GLB1
CRTAP
SUSD5
UBP1
CLASP2
PDCD6IP
STAC
TRANK1
EPM2AIP1
MLH1
LRRFIP2
ITGA9
CTDSPL
VILL
PLCD1
ACAA1
MYD88
OXSR1
XYLB
ACVR2B
WDR48
GORASP1
CSRNP1
SLC25A38
EIF1B
ZNF619
ZNF620
ZNF621
CTNNB1
ULK4
TRAK1
VIPR1
SEC22C
SS18L2
NKTR
ZBTB47
HIGD1A
GASK1A
POMGNT2
SNRK
ANO10
ABHD5
TCAIM
ZNF445
ZNF852
ZKSCAN7
ZNF660
ZNF197
ZNF35
ZNF502
ZNF501
KIAA1143
KIF15
TMEM42
ZDHHC3
EXOSC7
CDCP1
TMEM158
LARS2
LIMD1
SACM1L
LZTFL1
FYCO1
LRRC2
TDGF1
CCDC12
NBEAL2
KIF9
KLHL18
PTPN23
SCAP
ELP6
CSPG5
SMARCC1
DHX30
MAP4
CDC25A
ZNF589
NME6
PLXNB1
CCDC51
TMA7
ATRIP
SHISA5
PFKFB4
COL7A1
UQCRC1
SLC26A6
CELSR3
NCKIPSD
IP6K2
PRKAR2A
SLC25A20
ARIH2
P4HTM
WDR6
DALRD3
NDUFAF3
IMPDH2
QRICH1
QARS
USP19
LAMB2
CCDC71
KLHDC8B
USP4
RHOA
TCTA
AMT
NICN1
DAG1
BSN
APEH
MST1
GMPPB
IP6K1
TRAIP
CAMKV
MON1A
RBM6
RBM5
SEMA3F
GNAI2
IFRD2
NAA80
HYAL2
TUSC2
RASSF1
NPRL2
CYB561D2
TMEM115
CACNA2D2
HEMK1
CISH
MAPKAPK3
DOCK3
MANF
RBM15B
DCAF1
RAD54L2
TEX264
RRP9
ABHD14A
ACY1
DUSP7
POC1A
ALAS1
TWF2
WDR82
GLYCTK
BAP1
SEMA3G
TNNC1
NISCH
NT5DC2
PBRM1
GNL3
GLT8D1
SPCS1
NEK4
MUSTN1
STIMATE
SFMBT1
RFT1
PRKCD
DCP1A
CHDH
IL17RB
ACTR8
SELENOK
CACNA2D3
WNT5A
ERC2
CCDC66
TASOR
ARHGEF3
IL17RD
HESX1
APPL1
PDE12
ARF4
DENND6A
SLMAP
FLNB
ABHD6
RPP14
PXK
PDHB
KCTD6
C3orf67
PTPRG
C3orf14
CADPS
THOC7
ATXN7
PSMD6
PRICKLE2
MAGI1
SLC25A26
LRIG1
KBTBD8
SUCLG2
TAFA4
EOGT
TMF1
UBA3
ARL6IP5
FOXP1
EIF4E3
GPR27
RYBP
SHQ1
GXYLT2
PPP4R2
EBLN2
PDZRN3
ZNF717
ROBO1
GBE1
CADM2
VGLL3
CHMP2B
CGGBP1
ZNF654
C3orf38
ARL13B
DHFR2
NSUN3
ARL6
CRYBG3
RIOX2
CLDND1
CPOX
ST3GAL6
DCBLD2
CMSS1
TBC1D23
NIT2
TOMM70
LNP1
TMEM45A
TFG
SENP7
TRMT10C
ZBTB11
CEP97
NXPE3
NFKBIZ
ALCAM
CBLB
BBX
CD47
IFT57
CIP2A
DZIP3
DPPA2
DPPA4
NECTIN3
PHLDB2
ABHD10
TAGLN3
C3orf52
CD200
ATG3
SLC35A5
CCDC80
GTPBP8
NEPRO
BOC
SPICE1
USF3
NAA50
ATP6V1A
ZDHHC23
QTRT2
GAP43
LSAMP
B4GALT4
TMEM39A
POGLUT1
TIMMDC1
ADPRH
COX17
NR1I2
GSK3B
LRRC58
FSTL1
NDUFB4
RABL3
GTF2E1
STXBP5L
POLQ
GOLGB1
IQCB1
EAF2
SLC15A2
FAM162A
WDR5B
KPNA1
PARP9
DTX3L
PARP14
HSPBAP1
SLC49A4
SEMA5B
PDIA5
SEC22A
ADCY5
HACD2
MYLK
CCDC14
KALRN
UMPS
ITGB5
HEG1
SLC12A8
ZNF148
SNX4
OSBPL11
SLC41A3
KLF15
ZXDC
TXNRD3
CHCHD6
PLXNA1
TPRA1
MCM2
PODXL2
SEC61A1
RUVBL1
EEFSEC
RPN1
RAB7A
ACAD9
ISY1
CNBP
COPG1
HMCES
H1FX
MBD4
IFT122
PLXND1
TMCC1
PIK3R4
ATP2C1
ASTE1
NUDT16
DNAJC13
ACAD11
UBA5
NPHP3
TMEM108
CDV3
TOPBP1
SRPRB
RAB6B
RYK
AMOTL2
ANAPC13
CEP63
EPHB1
PPP2R3A
MSL2
PCCB
STAG1
SLC35G2
NCK1
DZIP1L
DBR1
ARMC8
MRAS
ESYT3
CEP70
FAIM
PIK3CB
COPB2
RBP1
SLC25A36
SPSB4
PXYLP1
ZBTB38
RASA2
RNF7
ATP1B3
TFDP2
GK5
XRN1
ATR
PLS1
TRPC1
PCOLCE2
U2SURP
CHST2
DIPK2A
PLOD2
PLSCR1
GYG1
HLTF
HPS3
TM4SF18
WWTR1
COMMD2
RNF13
PFN2
TSC22D2
SERP1
EIF2A
SELENOT
SIAH2
MED12L
MBNL1
P2RY1
RAP2B
ARHGEF26
DHX36
MME
PLCH1
C3orf33
SLC33A1
GMPS
SSR3
TIPARP
CCNL1
VEPH1
RSRC1
MLF1
GFM1
LXN
MFSD1
IFT80
SMC4
TRIM59
KPNA4
PPM1L
B3GALNT1
NMD3
SPTSSB
PDCD10
SERPINI1
GOLIM4
ACTRT3
MYNN
LRRC34
SEC62
GPR160
PHC3
PRKCI
SKIL
EIF5A2
TNIK
FNDC3B
NCEH1
ECT2
NLGN1
TBL1XR1
ZMAT3
PIK3CA
KCNMB3
ZNF639
MFN1
GNB4
ACTL6A
NDUFB5
USP13
PEX5L
TTC14
FXR1
SOX2
ATP11B
DCUN1D1
MCCC1
MCF2L2
B3GNT5
KLHL24
YEATS2
PARL
ABCC5
EIF2B5
DVL3
AP2M1
ABCF3
ALG3
EEF1AKMT4
CAMK2N2
PSMD2
EIF4G1
FAM131A
CLCN2
POLR2H
MAGEF1
VPS8
MAP3K13
TMEM41A
SENP2
IGF2BP2
TRA2B
ETV5
TBCCD1
DNAJB11
EIF4A2
RFC4
ST6GAL1
LPP
P3H2
IL1RAP
CCDC50
FGF12
MB21D2
PLAAT1
OPA1
HES1
ATP13A3
TMEM44
LSG1
XXYLT1
ACAP2
TNK2
TFRC
SLC51A
PCYT1A
TCTEX1D2
UBXN7
RNF168
WDR53
FBXO45
NRROS
PIGX
CEP19
PAK2
SENP5
NCBP2
NCBP2AS2
DLG1
BDH1
RUBCN
FYTTD1
LRCH3
IQCG
ZNF595
ZNF718
ZNF732
ZNF141
PIGG
PDE6B
ATP5ME
PCGF3
GAK
TMEM175
DGKQ
FGFRL1
MAEA
UVSSA
SLBP
TACC3
TMEM129
LETM1
NSD2
NELFA
C4orf48
NAT8L
HAUS3
MXD4
ZFYVE28
RNF4
FAM193A
TNIP2
SH3BP2
ADD1
MFSD10
NOP14
HTT
RGS12
LRPAP1
ADRA2C
TMEM128
LYAR
ZBTB49
NSG1
STK32B
EVC2
EVC
CRMP1
WFS1
PPP2R2C
MAN2B2
AC093323.1
MRFAP1L1
BLOC1S4
KIAA0232
TBC1D14
TADA2B
GRPEL1
AFAP1
HTRA3
ACOX3
TRMT44
WDR1
ZNF518B
RAB28
BOD1L1
CC2D2A
FBXL5
FAM200B
PROM1
TAPT1
LDB2
QDPR
LAP3
MED28
DCAF16
NCAPG
LCORL
SLIT2
PACRGL
ADGRA3
DHX15
SOD3
CCDC149
LGI2
SEPSECS
PI4K2B
ZCCHC4
ANAPC4
SLC34A2
SEL1L3
SMIM20
RBPJ
TBC1D19
STIM2
PCDH7
C4orf19
RELL1
PGM2
TBC1D1
KLF3
FAM114A1
KLHL5
WDR19
RFC1
LIAS
UGDH
SMIM14
UBE2K
PDS5A
N4BP2
RHOH
CHRNA9
RBM47
NSUN7
APBB2
UCHL1
TMEM33
SLC30A9
BEND4
ATP8A1
GUF1
GNPDA2
COMMD8
ATP10D
NFXL1
NIPAL1
TEC
SLAIN2
SLC10A4
FRYL
OCIAD1
OCIAD2
DCUN1D4
SGCB
USP46
ERVMER34-1
FIP1L1
CHIC2
KIT
KDR
SRD5A3
TMEM165
CLOCK
NMU
EXOC1
CEP135
KIAA1211
AASDH
PPAT
PAICS
SRP72
REST
NOA1
POLR2B
ADGRL3
CENPC
UBA6
YTHDC1
UTP3
RUFY3
GRSF1
MOB1B
DCK
SLC4A4
ADAMTS3
COX18
ANKRD17
AFP
CXCL6
CXCL5
CXCL3
MTHFD2L
PARM1
RCHY1
THAP6
CDKL2
G3BP2
USO1
NAAA
SDAD1
NUP54
SCARB2
SHROOM3
SEPTIN11
CCNI
CCNG2
CNOT6L
FRAS1
ANXA3
BMP2K
PAQR3
RASGEF1B
HNRNPD
HNRNPDL
ENOPH1
SEC31A
THAP9
LIN54
COPS4
COQ2
HPSE
HELQ
ABRAXAS1
GPAT3
CDS1
WDFY3
MAPK10
PTPN13
AFF1
KLHL8
HSD17B11
NUDT9
SPP1
PKD2
ABCG2
PPM1K
HERC6
HERC5
PYURF
HERC3
FAM13A
TIGD2
SNCA
GRID2
SMARCAD1
RAP1GDS1
TSPAN5
EIF4E
ADH5
TRMT10A
LAMTOR3
DNAJB14
H2AFZ
DDIT4L
PPP3CA
SLC39A8
NFKB1
MANBA
UBE2D3
CISD2
SLC9B2
BDH2
CENPE
TACR3
PPA2
INTS12
GSTCD
TBCK
AIMP1
PAPSS1
CYP2U1
HADH
OSTC
COL25A1
SEC24B
MCUB
CASP6
PLA2G12A
GAR1
ELOVL6
FAM241A
AP1AR
TIFA
ZGRF1
LARP7
ANK2
CAMK2D
ARSJ
UGT8
TRAM1L1
PRSS12
METTL14
SEC24D
SYNPO2
USP53
C4orf3
PDE5A
MAD2L1
PRDM5
ANXA5
EXOSC9
CCNA2
BBS7
KIAA1109
BBS12
FGF2
SPATA5
SPRY1
ANKRD50
INTU
HSPA4L
PLK4
MFSD8
ABHD18
LARP1B
PGRMC2
JADE1
SCLT1
C4orf33
PCDH10
PABPC4L
PCDH18
SLC7A11
NOCT
ELF2
NDUFC1
NAA15
RAB33B
MGST2
MAML3
CLGN
ELMOD2
RNF150
ZNF330
USP38
GAB1
SMARCA5
ANAPC10
ABCE1
OTUD4
SMAD1
MMAA
ZNF827
LSM6
SLC10A7
TMEM184C
PRMT9
ARHGAP10
LRBA
SH3D19
FAM160A1
GATB
FBXW7
TMEM154
TIGD4
ARFIP1
FHDC1
TRIM2
MND1
TMEM131L
RNF175
SFRP2
PLRG1
LRAT
MAP9
GUCY1B1
TMEM144
C4orf46
ETFDH
PPID
FNIP2
RAPGEF2
NAF1
NPY1R
NPY5R
TMA16
APELA
TRIM61
FAM218A
TMEM192
KLHL2
MSMO1
CPE
PALLD
CBR4
SH3RF1
NEK1
CLCN3
HPF1
MFAP3L
AADAT
GALNT7
HMGB2
SAP30
FBXO8
CEP44
WDR17
SPCS3
VEGFC
NEIL3
AGA
TENM3
DCTD
WWC2
CDKN2AIP
ING2
RWDD4
TRAPPC11
STOX2
IRF2
CASP3
PRIMPOL
CENPU
ACSL1
SLC25A4
CFAP97
SNX25
UFSP2
C4orf47
CCDC110
SORBS2
FAM149A
KLKB1
FAT1
ZFP42
TRIML2
FRG1
PLEKHG4B
CCDC127
SDHA
PDCD6
EXOC3
CEP72
BRD9
TRIP13
NKD2
SLC12A7
TERT
CLPTM1L
LPCAT1
NDUFS6
IRX2
ADAMTS16
MED10
NSUN2
SRD5A1
TENT4A
ADCY2
MTRR
FASTKD3
SEMA5A
ATPSCKMT
CMBL
MARCH6
ANKRD33B
DAP
CTNND2
TRIO
OTULINL
OTULIN
ANKH
FBXL7
ZNF622
RETREG1
MYO10
BASP1
DROSHA
C5orf22
PDZD2
GOLPH3
MTMR12
ZFR
SUB1
ADAMTS12
AMACR
RAI14
RAD1
BRIX1
DNAJC21
PRLR
UGT3A2
LMBRD2
SKP2
NADK2
SLC1A3
NIPBL
CPLANE1
NUP155
WDR70
LIFR
OSMR
RICTOR
DAB2
TTC33
PRKAA1
OXCT1
C5orf51
SELENOP
ZNF131
HMGCS1
TMEM267
C5orf34
PAIP1
NNT
HCN1
EMB
PARP8
PELO
ITGA1
ITGA2
MOCS2
FST
NDUFS4
ARL15
GPX8
DHX29
MTREX
PLPP1
SLC38A9
MAP3K1
SETD9
MIER3
GPBP1
PLK2
RAB3C
PDE4D
DEPDC1B
ELOVL7
ERCC8
NDUFAF2
SMIM15
ZSWIM6
KIF2A
DIMT1
IPO11
RNF180
RGS7BP
SHISAL2B
SREK1IP1
CWC27
CENPK
PPWD1
TRIM23
SHLD3
TRAPPC13
SGTB
NLN
ERBIN
MAST4
PIK3R1
SLC30A5
CCNB1
CENPH
CDK7
CCDC125
AK6
TAF9
RAD17
MARVELD2
OCLN
GTF2H2C
GTF2H2
BDP1
MCCC2
MAP1B
PTCD2
TNPO1
FCHO2
BTF3
ANKRA2
UTP15
ENC1
HEXB
GFM2
NSA2
FAM169A
GCNT4
HMGCR
CERT1
POLK
POC5
IQGAP2
F2R
F2RL1
AGGF1
ZBED3
WDR41
AP3B1
SCAMP1
LHFPL2
ARSB
JMY
HOMER1
TENT2
MTX3
THBS4
SERINC5
ZFYVE16
ANKRD34B
DHFR
MSH3
RASGRF2
ZCCHC9
SSBP2
ATG10
TMEM167A
XRCC4
VCAN
EDIL3
COX7C
RASA1
CCNH
TMEM161B
CETN3
POLR3G
LYSMD3
ADGRV1
ARRDC3
FAM172A
SLF1
MCTP1
TTC37
ARSK
RHOBTB3
GLRX
ELL2
PCSK1
CAST
ERAP1
LNPEP
RIOK2
RGMB
CHD1
FAM174A
SLCO4C1
PAM
GIN1
PPIP5K2
C5orf30
NUDT12
EFNA5
FBXL17
FER
PJA2
MAN2A1
SLC25A46
WDR36
CAMK4
STARD4
NREP
APC
SRP19
REEP5
DCP2
MCC
YTHDC2
KCNN2
TRIM36
PGGT1B
CCDC112
FEM1C
TICAM2
TMED7
CDO1
ATG12
COMMD10
SEMA6A
DTWD2
DMXL1
TNFAIP8
HSD17B4
SRFBP1
LOX
SNCAIP
SNX2
SNX24
PPIC
CEP120
CSNK1G3
ZNF608
GRAMD2B
ALDH7A1
PHAX
LMNB1
MARCH3
MEGF10
PRRC1
SLC12A2
FBN2
SLC27A6
ISOC1
ADAMTS19
HINT1
LYRM7
CDC42SE2
RAPGEF6
FNIP1
P4HA2
PDLIM4
SLC22A5
RAD50
KIF3A
SEPTIN8
UQCRQ
AFF4
HSPA4
FSTL4
C5orf15
VDAC1
TCF7
SKP1
PPP2CA
CDKL3
UBE2B
CDKN2AIPNL
JADE2
SAR1B
SEC24A
CAMLG
DDX46
C5orf24
TXNDC15
H2AFY
TGFBI
SMAD5
SPOCK1
KLHL3
HNRNPA0
FAM13B
BRD8
KIF20A
CDC23
CDC25C
FAM53C
KDM3B
REEP2
EGR1
ETF1
HSPA9
CTNNA1
SIL1
PAIP2
PROB1
SPATA24
DNAJC18
ECSCR
UBE2D2
CXXC5
CYSTM1
PFDN1
HBEGF
ANKHD1
SRA1
APBB3
SLC35A4
NDUFA2
TMCO6
IK
WDR55
HARS
HARS2
ZMAT2
PCDHA4
PCDHA10
PCDHA12
PCDHA13
PCDHAC1
PCDHB2
PCDHB3
PCDHB5
PCDHB16
PCDHB9
PCDHB10
PCDHB12
PCDHB13
PCDHB14
PCDHB15
TAF7
PCDHGA1
PCDHGA5
PCDHGA6
PCDHGA8
PCDHGB6
PCDHGA10
PCDHGB7
PCDHGA12
PCDHGC3
DIAPH1
HDAC3
RELL2
FCHSD1
ARAP3
PCDH1
DELE1
RNF14
GNPDA1
NDFIP1
SPRY4
ARHGAP26
NR3C1
YIPF5
PRELID2
PLAC8L1
LARS
RBM27
TCERG1
PPP2R2B
DPYSL3
JAKMIP2
SCGB3A2
SPINK5
FBXO38
AFAP1L1
GRPEL2
PCYOX1L
CSNK1A1
PPARGC1B
SLC26A2
HMGXB3
PDGFRB
ARSI
TCOF1
CD74
NDST1
MYOZ3
RBM22
DCTN4
SMIM3
ZNF300
GPX3
TNIP1
ANXA6
CCDC69
GM2A
SLC36A1
SPARC
ATOX1
GRIA1
FAM114A2
MFAP3
GALNT10
SAP30L
LARP1
FAXDC2
CNOT8
GEMIN5
MED7
CYFIP2
ADAM19
THG1L
LSM11
CLINT1
RNF145
UBLCP1
TTC1
PWWP2A
CCNJL
SLU7
PTTG1
GABRA1
CCNG1
NUDCD2
HMMR
MAT2B
WWC1
RARS
FBLL1
PANK3
SPDL1
DOCK2
RANBP17
FBXW11
STK10
UBTD2
SH3PXD2B
NEURL1B
DUSP1
ERGIC1
ATP6V0E1
CREBRF
BNIP1
STC2
BOD1
CPEB4
SFXN1
HRH2
SIMC1
KIAA1191
ARL10
NOP16
HIGD2A
CLTB
FAF2
RNF44
GPRIN1
TSPAN17
UNC5A
UIMC1
ZNF346
FGFR4
NSD1
LMAN2
RGS14
GRK6
PRR7
DBN1
PDLIM7
DDX41
FAM193B
TMED9
B4GALT7
N4BP3
RMND5B
NHP2
HNRNPAB
PHYKPL
COL23A1
CLK4
ZNF354A
ZNF354B
ZNF454
ZNF879
ZNF354C
ADAMTS2
RUFY1
CANX
MAML1
MGAT4B
SQSTM1
MRNIP
TBC1D9B
RNF130
MAPK9
GFPT2
CNOT6
FLT4
MGAT1
ZFP62
TRIM41
RACK1
TRIM52
DUSP22
EXOC2
GMDS
WRNIP1
SERPINB1
SERPINB9
SERPINB6
NQO2
RIPK1
BPHL
TUBB2A
TUBB2B
PSMG4
SLC22A23
FAM50B
PRPF4B
ECI2
CDYL
RPP40
LYRM4
FARS2
RREB1
SSR1
RIOK1
DSP
SNRNP48
TXNDC5
BLOC1S5
EEF1E1
SLC35B3
GCNT2
C6orf52
PAK1IP1
TMEM14C
TMEM14B
SYCP2L
SMIM13
PHACTR1
TBC1D7
GFOD1
SIRT5
NOL7
RANBP9
MCUR1
RNF182
CD83
JARID2
DTNBP1
MYLIP
CAP2
FAM8A1
NUP153
KIF13A
TPMT
KDM1B
DEK
RNF144B
ID4
MBOAT1
E2F3
CDKAL1
SOX4
MRS2
ALDH5A1
TDP2
ACOT13
C6orf62
GMNN
RIPOR2
CARMIL1
TRIM38
HIST1H1A
HIST1H1C
HIST1H4C
HIST1H2BG
HIST1H3E
HIST1H1D
HIST1H2BH
HIST1H3G
BTN3A2
BTN2A2
BTN2A1
HMGN4
ABT1
ZNF322
HIST1H2AG
HIST1H4I
PRSS16
ZNF391
ZNF184
HIST1H3H
HIST1H1B
ZNF165
ZSCAN16
ZKSCAN8
ZSCAN9
ZKSCAN4
ZSCAN26
PGBD1
ZSCAN31
ZKSCAN3
ZSCAN12
ZBED9
TRIM27
ZNF311
GABBR1
ZNRD1
PPP1R11
TRIM26
TRIM39
RPP21
GNL1
PRR3
ABCF1
PPP1R10
ATAT1
C6orf136
DHX16
PPP1R18
NRM
MDC1
TUBB
FLOT1
IER3
DDR1
GTF2H4
VARS2
CCHCR1
TCF19
POU5F1
MICA
DDX39B
NFKBIL1
AIF1
PRRC2A
BAG6
C6orf47
GPANK1
CSNK2B
ABHD16A
DDAH2
CLIC1
MSH5
SAPCD1
VARS
LSM2
HSPA1B
SNHG32
NEU1
EHMT2
ZBTB12
NELFE
SKIV2L
DXO
STK19
TNXB
ATF6B
FKBPL
PPT2
AGPAT1
RNF5
PBX2
GPSM3
TAP2
PSMB8
TAP1
BRD2
RXRB
SLC39A7
HSD17B8
RING1
VPS52
WDR46
PFDN6
RGL2
TAPBP
ZBTB22
DAXX
KIFC1
PHF1
CUTA
SYNGAP1
ZBTB9
BAK1
ITPR3
UQCC2
LEMD2
GRM4
HMGA1
NUDT3
PACSIN1
ILRUN
SNRPC
UHRF1BP1
TAF11
ANKS1A
ZNF76
DEF6
PPARD
FANCE
TEAD3
TULP1
FKBP5
SRPK1
MAPK14
MAPK13
BRPF3
KCTD20
STK38
SRSF3
PPIL1
C6orf89
MTCH1
PIM1
TBC1D22B
RNF8
CMTR1
CCDC167
ZFAND3
BTBD9
GLO1
GLP1R
KCNK5
DAAM2
OARD1
NFYA
FOXP4
MDFI
FRS3
TOMM6
USP49
MED20
BYSL
CCND3
TAF8
C6orf132
TRERF1
UBR2
TBCC
BICRAL
CNPY3
PEX6
PPP2R5D
MEA1
KLHDC3
RRP36
CUL7
KLC4
PTK7
SRF
DNPH1
TTBK1
CRIP3
ZNF318
POLR1C
YIPF3
XPO5
POLH
GTPBP2
MAD2L1BP
VEGFA
TMEM63B
SLC29A1
HSP90AB1
SLC35B2
NFKBIE
TMEM151B
AARS2
CDC5L
SUPT3H
ENPP4
ENPP5
RCAN2
PLA2G7
TNFRSF21
CD2AP
MMUT
CENPQ
C6orf141
MCM3
PAQR8
EFHC1
TRAM2
TMEM14A
GSTA4
ICK
FBXO9
ELOVL5
GCLC
LRRC1
FAM83B
COL21A1
DST
KIAA1586
ZNF451
BAG2
RAB23
PRIM2
KHDRBS2
AL135905.2
PHF3
LMBRD1
COL9A1
FAM135A
SDHAF4
SMAP1
OGFRL1
KHDC1
CGAS
MTO1
SLC17A5
CD109
COX7A2
TMEM30A
SENP6
MYO6
IRAK1BP1
PHIP
HMGN3
LCA5
SH3BGRL2
ELOVL4
TTK
BCKDHB
TENT5A
IBTK
TPBG
UBE3D
DOP1A
PGM3
RWDD2A
ME1
PRSS35
RIPPLY2
CYB5R4
MRAP2
CEP162
SNX14
SYNCRIP
ZNF292
SMIM8
C6orf163
SLC35A1
RARS2
ORC3
AKIRIN2
RNGTT
PNRC1
SRSF12
PM20D2
UBE2J1
RRAGD
ANKRD6
LYRM2
MDN1
CASP8AP2
MAP3K7
EPHA7
MANEA
FUT9
UFL1
GPR63
NDUFAF4
MMS22L
FBXL4
FAXC
COQ3
PNISR
USP45
CCNC
ASCC3
GRIK2
HACE1
LIN28B
BVES
POPDC3
PREP
ATG5
CRYBG1
RTN4IP1
QRSL1
CD24
MTRES1
BEND3
PDSS2
SEC63
OSTM1
SNX3
AFG1L
FOXO3
SESN1
CD164
SMPD2
MICAL1
ZBTB24
AK9
FIG4
WASF1
CDC40
CDK19
AMD1
GTF3C6
RPF2
SLC16A10
MFSD4B
REV3L
TRAF3IP2
FYN
TUBE1
FAM229B
MARCKS
HDAC2
HS3ST5
NT5DC1
TSPYL4
DSE
TSPYL1
CALHM6
RWDD1
ZUP1
KPNA5
DCBLD1
GOPC
NUS1
SLC35F1
CEP85L
MCM9
ASF1A
FAM184A
MAN1A1
TBC1D32
GJA1
HSF2
SERINC1
FABP7
SMPDL3A
TRDN
RNF217
TPD52L1
HDDC2
HEY2
NCOA7
HINT3
TRMT11
CENPW
RNF146
ECHDC1
KIAA0408
SOGA3
PTPRK
ARHGAP18
L3MBTL3
SAMD3
TMEM200A
EPB41L2
AKAP7
MED23
ENPP1
CCN2
MOXD1
STX7
SLC18B1
TBPL1
SLC2A12
SGK1
HBS1L
MYB
AHI1
MTFR2
BCLAF1
MAP7
MAP3K5
PEX7
IL20RA
IFNGR1
PERP
HEBP2
NHSL1
CCDC28A
REPS1
ABRACL
HECA
CITED2
VTA1
ADGRG6
HIVEP2
AIG1
ADAT2
PEX3
FUCA2
PHACTR2
LTV1
SF3B5
UTRN
EPM2A
FBXO30
SHPRH
RAB32
STXBP5
SASH1
UST
TAB2
PPIL4
GINM1
KATNA1
LATS1
NUP43
PCMT1
LRP11
MTHFD1L
AKAP12
ZBTB2
RMND1
ARMT1
FBXO5
MTRF1L
RGS17
CNKSR3
SCAF8
TIAM2
TFB1M
ARID1B
TMEM242
ZDHHC14
SNX9
SYNJ2
SERAC1
GTF2H5
TULP4
TMEM181
DYNLT1
EZR
RSPH3
SOD2
ACAT2
TCP1
IGF2R
SLC22A3
MAP3K4
AGPAT4
QKI
PDE10A
SFT2D1
MPC1
RNASET2
FGFR1OP
AFDN
DACT2
SMOC2
THBS2
WDR27
C6orf120
PHF10
TCTE3
ERMARD
FAM120B
PSMB1
TBP
PDCD2
FAM20C
PDGFA
PRKAR1B
DNAAF5
SUN1
COX19
C7orf50
ZFAND2A
MICALL2
INTS1
MAFK
PSMG3
MAD1L1
MRM2
NUDT1
SNX8
EIF3B
CHST12
LFNG
BRAT1
IQCE
TTYH3
GNA12
CARD11
SDK1
FOXK1
AP5Z1
RBAK
WIPI2
FBXL18
ACTB
FSCN1
RNF216
PMS2
AIMP2
EIF2AK1
USP42
CYTH3
RAC1
DAGLB
KDELR2
ZDHHC4
C7orf26
ZNF316
ZNF12
C1GALT1
MIOS
RPA3
UMAD1
GLCCI1
ICA1
NDUFA4
PHF14
TMEM106B
VWDE
ARL4A
ETV1
DGKB
ANKMY2
BZW2
TSPAN13
AHR
SNX13
HDAC9
TWISTNB
MACC1
ITGB8
SP4
DNAH11
CDCA7L
RAPGEF5
TOMM7
FAM126A
KLHL7
NUP42
MALSU1
IGF2BP3
TRA2A
CCDC126
FAM221A
NPY
MPP6
GSDME
OSBPL3
CYCS
NFE2L3
HNRNPA2B1
CBX3
SNX10
SKAP2
HIBADH
TAX1BP1
JAZF1
CPVL
WIPF3
SCRN1
FKBP14
PLEKHA8
MTURN
ZNRF2
GGCT
GARS
MINDY4
ADCYAP1R1
PPP1R17
LSM5
AVL9
KBTBD2
FKBP9
NT5C3A
RP9
BBS9
DPY19L1
HERPUD2
SEPTIN7
EEPD1
ANLN
ELMO1
EPDR1
STARD3NL
AMPH
VPS41
POU6F2
YAE1
RALA
CDK13
MPLKIP
SUGCT
GLI3
C7orf25
PSMA2
HECW1
STK17A
COA1
BLVRA
URGCP
DBNL
POLM
AEBP1
POLD2
GCK
YKT6
NUDCD3
DDX56
TMED4
OGDH
ZMIZ2
PPIA
PURB
CCM2
TBRG4
ADCY1
IGFBP3
TNS3
HUS1
UPP1
FIGNL1
GRB10
COBL
SEC61G
EGFR
LANCL2
VOPP1
ZNF713
NIPSNAP2
PSPH
CCT6A
SUMF2
CHCHD2
ZNF736
ZNF680
ZNF107
ZNF138
ZNF273
ZNF117
ZNF92
VKORC1L1
GUSB
ASL
CRCP
TPST1
KCTD7
RABGEF1
TMEM248
SBDS
TYW1
AUTS2
GALNT17
CALN1
POM121
NSUN5
BAZ1B
BCL7B
TBL2
MLXIPL
DNAJC30
BUD23
STX1A
ABHD11
CLDN3
CLDN4
METTL27
ELN
LIMK1
EIF4H
RFC2
CLIP2
GTF2IRD1
GTF2I
CASTOR2
RCC1L
HIP1
RHBDD2
POR
TMEM120A
STYXL1
MDH2
HSPB1
YWHAG
SSC4D
DTX2
UPK3B
PTPN12
RSBN1L
TMEM60
PHTF2
MAGI2
GNAI1
CACNA2D1
PCLO
SEMA3A
SEMA3D
KIAA1324L
DMTF1
TMEM243
CROT
SLC25A40
DBF4
ADAM22
SRI
STEAP1
STEAP2
CFAP69
GTPBP10
CLDN12
CDK14
MTERF1
AKAP9
CYP51A1
KRIT1
ANKIB1
GATAD1
PEX1
FAM133B
CDK6
VPS50
TFPI2
GNG11
BET1
COL1A2
CASD1
SGCE
PEG10
PPP1R9A
PON2
SLC25A13
SEM1
SDHAF3
TAC1
LMTK2
TECPR1
BRI3
BAIAP2L1
NPTX2
TMEM130
TRRAP
SMURF1
ARPC1A
ARPC1B
BUD31
PTCD1
CPSF4
ATP5MF
ZNF789
ZNF394
ZKSCAN5
FAM200A
ZNF655
ZSCAN25
GJC3
ZKSCAN1
ZSCAN21
ZNF3
COPS6
MCM7
AP4M1
TAF6
CNPY4
LAMTOR4
GPC2
CASTOR3
PILRB
MEPCE
PPP1R35
TSC22D4
NYAP1
LRCH4
PCOLCE
MOSPD3
GNB2
GIGYF1
POP7
EPHB4
SLC12A9
TRIP6
SRRT
MUC3A
TRIM56
SERPINE1
AP1S1
VGF
PLOD3
ZNHIT1
CLDN15
FIS1
IFT22
COL26A1
CUX1
SH2B2
ORAI2
ALKBH4
LRWD1
FAM185A
FBXL13
LRRC17
ARMC10
NAPEPLD
PMPCB
DNAJC2
PSMC2
ORC5
KMT2E
SRPK2
PUS7
RINT1
EFCAB10
ATXN7L1
SYPL1
NAMPT
CCDC71L
PRKAR2B
HBP1
COG5
DUS4L
BCAP29
CBLL1
DLD
LAMB1
NRCAM
PNPLA8
THAP5
DNAJB9
IMMP2L
DOCK4
ZNF277
IFRD1
TMEM168
BMT2
SMIM30
MDFIC
TES
CAV1
MET
CAPZA2
ST7
CTTNBP2
LSM8
KCND2
TSPAN12
ING3
FAM3C
PTPRZ1
AASS
CADPS2
NDUFA5
WASL
GPR37
POT1
GCC1
ARF5
SND1
LRRC4
RBM28
HILPDA
METTL2B
FAM71F1
CALU
CCDC136
FLNC
KCP
ATP6V1F
TNPO3
TSPAN33
SMO
AHCYL2
STRIP2
NRF1
UBE2H
ZC3HC1
KLHDC10
TMEM209
CEP41
MEST
COPG2
MKLN1
PODXL
PLXNA4
CHCHD3
EXOC4
LRGUK
SLC35B4
AKR1B1
BPGM
CALD1
CYREN
WDR91
CNOT4
NUP205
STMP1
MTPN
PTN
CREB3L2
TRIM24
KIAA1549
ZC3HAV1L
ZC3HAV1
TTC26
UBN2
FMC1
LUC7L2
KLRG2
HIPK2
PARP12
KDM7A
SLC37A3
DENND2A
ADCK2
NDUFB2
BRAF
TMEM178B
AGK
DENND11
TAS2R4
TRBC2
EPHB6
GSTK1
CASP2
FAM131B
ZYX
EPHA1
TCAF1
ARHGEF5
CNTNAP2
CUL1
EZH2
PDIA4
ZNF786
ZNF425
ZNF398
ZNF282
ZNF212
ZNF783
ZNF777
ZBED6CL
REPIN1
AC073111.4
KCNH2
ABCB8
CDK5
SLC4A2
FASTK
TMUB1
AGAP3
ABCF2
CHPF2
NUB1
WDR86
RHEB
PRKAG2
GALNT11
KMT2C
XRCC2
ACTR3B
DPP6
PAXIP1
INSIG1
RBM33
LMBR1
NOM1
UBE3C
DNAJB6
PTPRN2
NCAPG2
ESYT2
WDR60
ZNF596
FBXO25
TDRP
ERICH1
CLN8
ARHGEF10
KBTBD11
MCPH1
AGPAT5
PRAG1
MFHAS1
ERI1
PPP1R3B
TNKS
MSRA
PINX1
XKR6
MTMR9
NEIL2
FDFT1
CTSB
LONRF1
DLC1
TUSC3
MICU3
ZDHHC2
VPS37A
MTMR7
SLC7A2
PDGFRL
MTUS1
PCM1
ASAH1
NAT1
PSD3
SH2D4A
INTS10
LPL
ATP6V1B2
LZTS1
GFRA2
XPO7
FAM160B2
REEP4
PHYHIP
POLR3D
SLC39A14
PPP3CC
SORBS3
PDLIM2
C8orf58
CCAR2
BIN3
RHOBTB2
TNFRSF10B
TNFRSF10D
TNFRSF10A
CHMP7
R3HCC1
LOXL2
ENTPD4
SLC25A37
STC1
NEFM
NEFL
DOCK5
KCTD9
CDCA2
PPP2R2A
BNIP3L
DPYSL2
TRIM35
PTK2B
EPHX2
CLU
SCARA3
ESCO2
PBK
ELP3
ZNF395
FBXO16
FZD3
EXTL3
INTS9
HMBOX1
KIF13B
DUSP4
SARAF
LEPROTL1
DCTN6
RBPMS
GTF2E2
GSR
UBXN8
PPP2CB
TEX15
PURG
WRN
NRG1
FUT10
TTI2
MAK16
RNF122
UNC5D
ERLIN2
PLPBP
ADGRA2
BRF2
RAB11FIP1
EIF4EBP1
ASH2L
STAR
LSM1
DDHD2
PLPP5
FGFR1
TACC1
TM2D2
ADAM9
IDO1
ZMAT4
SFRP1
GOLGA7
GINS4
GPAT4
KAT6A
AP3M2
IKBKB
POLB
VDAC3
SLC20A2
SMIM19
THAP1
RNF170
HOOK3
FNTA
HGSNAT
SPIDR
PRKDC
MCM4
UBE2V2
PCMTD1
RB1CC1
ATP6V1H
LYPLA1
TMEM68
TGS1
LYN
PLAG1
CHCHD7
IMPAD1
FAM110B
UBXN2B
SDCBP
NSMAF
TOX
CA8
RAB2A
CHD7
ASPH
GGH
YTHDF3
ARMC1
MTFR1
PDE7A
RRS1
MYBL1
VCPIP1
SGK3
COPS5
CSPP1
ARFGEF1
PREX2
C8orf34
SLCO5A1
PRDM14
NCOA2
TRAM1
LACTB2
TERF1
RDH10
STAU2
UBE2W
ELOC
TMEM70
JPH1
GDAP1
CRISPLD1
PEX2
PKIA
ZC2HC1A
TPD52
ZBTB10
ZNF704
FABP5
IMPA1
ZFAND1
CHMP4C
SNX16
LRRCC1
E2F5
RBIS
CA13
CA2
WWP1
RMDN1
CPNE3
MMP16
RIPK2
OSGIN2
NBN
DECR1
CALB1
TMEM64
C8orf88
PIP4P2
OTUD6B
LRRC69
RUNX1T1
TRIQK
RBM12B
TMEM67
PDP1
RAD54B
VIRMA
ESRP1
DPY19L4
INTS8
CCNE2
NDUFAF6
TP53INP1
PLEKHF2
C8orf37
UQCRB
MTERF3
PTDSS1
SDC2
MTDH
LAPTM4B
MATN2
ERICH5
RIDA
POP1
STK3
VPS13B
COX6C
POLR2K
ANKRD46
YWHAZ
ZNF706
GRHL2
RRM2B
UBR5
KLF10
AZIN1
ATP6V1C1
FZD6
SLC25A32
DCAF13
RIMS2
LRP12
OXR1
ANGPT1
EIF3E
EMC2
TMEM74
NUDCD1
ENY2
EBAG9
TRPS1
EIF3H
UTP23
RAD21
MED30
EXT1
SAMD12
MAL2
ENPP2
TAF2
DSCC1
DEPTOR
COL14A1
MTBP
SNTB1
HAS2
ZHX2
DERL1
TBC1D31
C8orf76
ZHX1
ATAD2
WDYHV1
FAM91A1
TMEM65
TRMT12
RNF139
TATDN1
NDUFB9
MTSS1
SQLE
WASHC5
NSMCE2
TRIB1
LRATD2
MYC
FAM49B
ASAP1
EFR3A
KCNQ3
LRRC6
PHF20L1
ST3GAL1
ZFAT
KHDRBS3
TRAPPC9
CHRAC1
AGO2
PTK2
SLC45A4
PTP4A3
TSNARE1
ADGRB1
JRK
LY6E
ZFP41
GLI4
ZNF696
TOP1MT
RHPN1
ZC3H3
GSDMD
NAPRT
TIGD5
PYCR3
TSTA3
ZNF623
ZNF707
MAPK15
FAM83H
SCRIB
PUF60
NRBP2
EPPK1
PLEC
GRINA
OPLAH
EXOSC4
GPAA1
CYC1
SHARPIN
MAF1
HGH1
MROH1
BOP1
SCX
HSF1
DGAT1
SLC52A2
FBXL6
CPSF1
VPS28
TONSL
CYHR1
FOXH1
PPP1R16A
MFSD3
RECQL4
LRRC14
ARHGAP39
ZNF251
ZNF34
ZNF7
COMMD5
ZNF250
ZNF16
C8orf33
KANK1
SMARCA2
VLDLR
PUM3
RFX3
SLC1A1
PLPP6
CDC37L1
RCL1
JAK2
RLN2
PLGRKT
RIC1
ERMP1
KIAA2026
RANBP6
UHRF2
GLDC
KDM4C
DMAC1
PTPRD
MPDZ
NFIB
ZDHHC21
TTC39B
SNAPC3
PSIP1
CCDC171
BNC2
CNTLN
SH3GL2
RRAGA
HAUS6
DENND4C
SLC24A2
MLLT3
FOCAD
HACD4
MTAP
DMRTA1
ELAVL2
CAAP1
PLAA
TEK
MOB3B
C9orf72
ACO1
DDX58
TOPORS
SMIM27
NDUFB6
APTX
DNAJA1
SMU1
B4GALT1
BAG1
CHMP5
NFX1
NOL6
ANKRD18B
UBE2R2
UBAP2
DCAF12
UBAP1
KIF24
NUDT2
C9orf24
FAM219A
ENHO
RPP25L
DCTN3
SIGMAR1
GALT
IL11RA
DNAJB5
VCP
FANCG
PIGO
STOML2
UNC13B
RUSC2
TESK1
SIT1
CCDC107
ARHGEF39
TPM2
TLN1
CREB3
GBA2
RGP1
NPR2
HINT2
TMEM8B
RECK
GLIPR2
CLTA
GNE
RNF38
MELK
ZCCHC7
GRHPR
ZBTB5
POLR1E
FBXO10
TOMM5
TRMT10B
EXOSC3
DCAF10
SLC25A51
SHB
ALDH1B1
IGFBPL1
ANKRD18A
FAM122A
FXN
FAM189A2
PTAR1
C9orf135
MAMDC2
SMC5
TRPM3
CEMIP2
ABHD17B
C9orf85
RORB
C9orf40
CARNMT1
NMRK1
OSTF1
PCSK5
RFK
PRUNE2
VPS13A
GNA14
GNAQ
CEP78
PSAT1
TLE4
TLE1
UBQLN1
GKAP1
KIF27
HNRNPK
RMI1
AGTPBP1
NAA35
GOLM1
ISCA1
TUT7
DAPK1
CTSL
CDK20
SPIN1
S1PR3
CKS2
SECISBP2
SEMA4D
AUH
NFIL3
SPTLC1
IARS
NOL8
CENPP
IPPK
BICD2
ZNF484
FGD3
SUSD3
CARD19
NINJ1
WNK2
FAM120AOS
FAM120A
PHF2
PTPDC1
ZNF169
MFSD14B
FANCC
PTCH1
ERCC6L2
SLC35D2
ZNF367
HABP4
CDC14B
PRXL2C
ZNF510
ZNF782
CTSV
CCDC180
TDRD7
TSTD2
NCBP1
XPA
TRMO
ANP32B
NANS
TRIM14
CORO2A
TBC1D2
ANKS6
GALNT12
TGFBR1
ALG2
SEC61B
STX17
ERP44
INVS
TEX10
MSANTD3
TMEFF1
CAVIN4
PLPPR1
ZNF189
TMEM246
RNF20
SMC2
NIPSNAP3A
ABCA1
SLC44A1
FSD1L
FKTN
TMEM38B
ZNF462
RAD23B
KLF4
ELP1
CTNNAL1
TMEM245
EPB41L4B
PALM2-AKAP2
LPAR1
ECPAS
ZNF483
PTGR1
DNAJC25
GNG10
UGCG
SUSD1
PTBP3
HSDL2
KIAA1958
INIP
SNX30
ZNF883
ZFP37
FKBP15
SLC31A1
CDC26
PRPF4
BSPRY
HDHD3
ALAD
POLE3
RGS3
ZNF618
KIF12
COL27A1
AKNA
WHRN
ATP6V1G1
TMEM268
TNC
ASTN2
TRIM32
BRINP1
CDK5RAP2
MEGF9
FBXW2
B3GNT10
PSMD5
PHF19
CNTRL
RAB14
GSN
STOM
DAB2IP
NDUFA8
RBM18
MRRF
PDCL
RC3H2
ZBTB6
ZBTB26
RABGAP1
STRBP
CRB2
DENND1A
NEK6
PSMB7
NR6A1
OLFML2A
ARPC5L
GOLGA1
SCAI
PPP6C
RABEPK
HSPA5
GAPVD1
MAPKAP1
PBX3
MVB12B
ZBTB34
RALGPS1
ANGPTL2
GARNL3
SLC2A8
LRSAM1
NIBAN2
STXBP1
PTRH1
CDK9
FPGS
ENG
AK1
ST6GALNAC6
ST6GALNAC4
PIP5KL1
DPM2
FAM102A
NAIF1
SLC25A25
PTGES2
C9orf16
DNM1
GOLGA2
SWI5
TRUB2
COQ4
SLC27A4
URM1
CERCAM
ODF2
GLE1
SPTAN1
WDR34
SET
PKN3
ZDHHC12
ZER1
TBC1D13
ENDOG
SPOUT1
KYAT1
LRRC8A
PHYHD1
DOLK
NUP188
SH3GLB2
MIGA2
DOLPP1
PTPA
NTMT1
ASB6
PRRX2
PTGES
TOR1B
TOR1A
C9orf78
USP20
FNBP1
GPR107
NCS1
HMCN2
ASS1
FUBP3
EXOSC2
ABL1
LAMC3
AIF1L
NUP214
FAM78A
PRRC2B
POMT1
UCK1
RAPGEF1
SETX
TTF1
DDX31
GTF3C4
TSC1
GTF3C5
RALGDS
SURF6
MED22
SURF1
SURF2
SURF4
REXO4
ADAMTS13
CACFD1
SLC2A6
SARDH
VAV2
BRD3OS
BRD3
RXRA
COL5A1
OLFM1
PPP1R26
C9orf116
CAMSAP1
UBAC1
NACC2
TMEM250
QSOX2
GPSM1
SNAPC4
ENTR1
PMPCA
INPP5E
SEC16A
NOTCH1
EGFL7
AGPAT2
DIPK1B
TMEM141
RABL6
PHPT1
EDF1
TRAF2
FBXW5
PAXX
ABCA2
NPDC1
SAPCD2
MAN1B1
DPP7
ANAPC2
SSNA1
TPRN
TMEM203
NDOR1
BX255925.3
RNF208
TUBB4B
NELFB
TOR4A
NRARP
NSMF
DPH7
ZMYND19
ARRDC1
EHMT1
CACNA1B
ZMYND11
DIP2C
LARP4B
GTPBP4
IDI1
WDR37
PFKP
PITRM1
KLF6
AKR1C3
NET1
ASB13
TASOR2
GDI2
ANKRD16
FBH1
RBM17
PFKFB3
PRKCQ
SFMBT2
ITIH5
KIN
ATP5F1C
CELF2
USP6NL
ECHDC3
UPF2
DHTKD1
SEC61A2
NUDT5
CDC123
CAMK1D
OPTN
MCM10
UCMA
PHYH
SEPHS1
BEND7
PRPF18
FRMD4A
FAM107B
HSPA14
HSPA14-1
SUV39H2
DCLRE1C
ACBD7
RPP38
NMT2
FAM171A1
MINDY3
PTER
RSU1
TRDMT1
VIM
STAM
CACNB2
NSUN6
ARL5B
PLXDC2
NEBL
MLLT10
DNAJC1
BMI1
PIP4K2A
MSRB2
KIAA1217
ARHGAP21
PRTFDC1
THNSL1
MYO3A
PDSS1
ABI1
ANKRD26
YME1L1
MASTL
ACBD5
RAB18
ARMC4
MPP7
WAC
BAMBI
JCAD
MTPAP
MAP3K8
ARHGAP12
KIF5B
EPC1
CCDC7
ITGB1
PARD3
CUL2
CREM
CCNY
FZD8
ZNF248
ZNF33A
ZNF33B
BMS1
RET
RASGEF1A
HNRNPF
ZNF239
ZNF485
ZNF32
CXCL12
RASSF4
ZNF22
MARCH8
ZFAND4
TIMM23
NCOA4
GPRIN2
ZNF488
MAPK8
ARHGAP22
ERCC6
OGDHL
PARG
SGMS1
PRKG1
CSTF2T
ZWINT
IPMK
CISD1
UBE2D1
TFAM
SLC16A9
CCDC6
ANK3
CDK1
RHOBTB1
RTKN2
ADO
NRBF2
JMJD1C
REEP3
DNAJC12
SIRT1
HERC4
HNRNPH3
RUFY2
DNA2
SLC25A16
TET1
CCAR1
STOX1
DDX50
DDX21
KIF1BP
VPS26A
SUPV3L1
HK1
TSPAN15
FAM241B
H2AFY2
AIFM2
TYSND1
PPA1
LRRC20
EIF4EBP2
NODAL
PALD1
SGPL1
PCBD1
UNC5B
SLC29A3
CDH23
PSAP
CHST3
ASCC1
ANAPC16
DDIT4
DNAJB12
MICU1
MCU
P4HA1
NUDT13
ECD
DNAJC9
ANXA7
MSS51
PPP3CB
USP54
SEC24C
FUT11
CHCHD1
ZSWIM8
CAMK2G
PLAU
VCL
AP3M1
ADK
KAT6B
SAMD8
VDAC2
COMTD1
KCNMA1
DLG5
POLR3A
ZMIZ1
PPIF
ZCCHC24
TMEM254
PLAC9
ANXA11
DYDC2
PRXL2A
TSPAN14
GHITM
CDHR1
CCSER2
WAPL
BMPR1A
SNCG
GLUD1
SHLD2
MINPP1
PAPSS2
ATAD1
PTEN
RNLS
STAMBPL1
LIPA
IFIT1
IFIT5
SLC16A12
PANK1
KIF20B
HTR7
RPP30
HECTD2
PPP1R3C
TNKS2
FGFBP3
BTAF1
MARCH5
IDE
KIF11
EXOC6
CEP55
FRA10AC1
SLC35G1
NOC3L
HELLS
PDLIM1
SORBS1
ALDH18A1
TCTN3
ENTPD1
CCNJ
ZNF518A
TM9SF3
PIK3AP1
LCOR
ARHGAP19
FRAT2
RRP12
PGAM1
EXOSC1
ZDHHC16
MMS19
UBTD1
MORN4
PI4K2A
AVPI1
MARVELD1
ZFYVE27
SFRP5
R3HCC1L
HPS1
GOT1
SLC25A28
ENTPD7
CUTC
COX15
DNMBP
ERLIN1
CHUK
CWF19L1
BLOC1S2
SCD
NDUFB8
HIF1AN
SLF2
TWNK
LZTS2
BTRC
DPCD
POLL
FBXW4
NPM3
OGA
KCNIP2
ARMH3
HPS6
LDB1
PPRC1
NOLC1
GBF1
NFKB2
FBXL15
CUEDC2
MFSD13A
ACTR1A
SUFU
TRIM8
ARL3
SFXN2
BORCS7
AS3MT
CNNM2
NT5C2
INA
PCGF6
TAF5
ATP5MD
PDCD11
SH3PXD2A
STN1
SLK
GSTO1
GSTO2
ITPRIP
SORCS3
SORCS1
XPNPEP1
ADD3
MXI1
SMNDC1
DUSP5
SMC3
PDCD4
BBIP1
SHOC2
GPAM
ZDHHC6
VTI1A
TCF7L2
CASP7
DCLRE1A
NHLRC2
CCDC186
VWA2
AFAP1L2
ABLIM1
FAM160B1
TRUB1
ATRNL1
CCDC172
C10orf82
SHTN1
SLC18A2
PDZD8
RAB11FIP2
FAM204A
CACUL1
EIF3A
DENND10
SFXN4
PRDX3
RGS10
TIAL1
BAG3
INPP5F
MCMBP
SEC23IP
WDR11
FGFR2
ATE1
NSMCE4A
TACC2
PLEKHA1
HTRA1
CUZD1
C10orf88
PSTK
IKZF5
ACADSB
BUB3
CPXM2
CHST15
OAT
NKX1-2
LHPP
FAM53B
EEF1AKMT2
ABRAXAS2
ZRANB1
CTBP2
EDRF1
UROS
BCCIP
DHX32
DOCK1
MKI67
MGMT
C10orf143
GLRX3
TCERG1L
PPP2R2D
DPYSL4
STK32C
PWWP2B
INPP5A
CFAP46
KNDC1
UTF1
TUBGCP2
ZNF511
FUOM
ECHS1
PAOX
MTG1
BET1L
RIC8A
SIRT3
PSMD13
PGGHG
IFITM2
IFITM3
B4GALNT4
PKP3
SIGIRR
ANO9
PTDSS2
RNH1
HRAS
RASSF7
PHRF1
DEAF1
EPS8L2
TMEM80
TALDO1
GATD1
CEND1
SLC25A22
PIDD1
PNPLA2
CRACR2B
CD151
POLR2L
TSPAN4
CHID1
AP2A2
TOLLIP
BRSK2
MOB2
CTSD
CD81
TSSC4
SLC22A18
NAP1L4
ZNF195
ART5
NUP98
PGAP2
RHOG
STIM1
RRM1
TRIM21
TRIM6
TRIM5
TRIM22
FAM160A2
CCKBR
CAVIN3
TRIM3
ARFIP2
TIMM10B
DNHD1
RRP8
ILK
TAF10
TPP1
DCHS1
ZNF215
ZNF214
SYT9
PPFIBP2
CYB5R2
TUB
RIC3
STK33
TRIM66
ST5
AKIP1
TMEM9B
NRIP3
DENND5A
TMEM41B
ZNF143
WEE1
SWAP70
SBF2
ADM
AMPD3
RNF141
MRVI1
CTR9
EIF4G2
ZBED5
GALNT18
USP47
DKK3
PARVA
TEAD1
RASSF10
ARNTL
BTBD10
FAR1
SPON1
RRAS2
COPB1
PSMA1
PDE3B
CYP2R1
C11orf58
PLEKHA7
PIK3C2A
NUCB2
KCNJ11
SERGEF
SAAL1
HPS5
GTF2H1
LDHC
TSG101
UEVLD
SPTY2D1OS
SPTY2D1
PTPN5
ZDHHC13
E2F8
NAV2
HTATIP2
PRMT3
ANO5
FANCF
SVIP
CCDC34
LGR4
LIN7C
BDNF
KIF18A
METTL15
ARL14EP
MPPED2
DNAJC24
IMMP1L
ELP4
RCN1
EIF3M
QSER1
DEPDC7
TCP11L1
CSTF3
HIPK3
CD59
FBXO3
LMO2
CAPRIN1
NAT10
ABTB2
CAT
APIP
PDHX
CD44
PAMR1
FJX1
TRIM44
LDLRAD3
COMMD9
PRR5L
TRAF6
C11orf74
API5
TTC17
HSD17B12
ALKBH3
EXT2
CD82
TSPAN18
TP53I11
SYT13
SLC35C1
CRY2
MAPK8IP1
LARGE2
PHF21A
CREB3L1
MDK
AMBRA1
HARBI1
ATG13
ARHGAP1
ZNF408
CKAP5
LRP4
C11orf49
ARFGAP2
PACSIN3
DDB2
ACP2
NR1H3
MADD
SLC39A13
PSMC3
CELF1
NDUFS3
PTPMT1
KBTBD4
MTCH2
FNBP4
NUP160
PTPRJ
LRRC55
TNKS1BP1
SSRP1
SLC43A3
RTN4RL2
SLC43A1
TIMM10
UBE2L6
SERPING1
ZDHHC5
MED19
TMX2
SELENOH
CTNND1
LPXN
ZFP91
GLYATL1
FAM111B
FAM111A
DTX4
OSBP
PATL1
STX3
CCDC86
PRPF19
TMEM109
TMEM132A
SLC15A3
VWCE
DDB1
TKFC
CYB561A3
TMEM138
TMEM216
CPSF7
SYT7
DAGLA
MYRF
TMEM258
FEN1
FADS2
FADS1
FADS3
RAB3IL1
FTH1
INCENP
ASRGL1
EEF1G
TUT1
MTA2
B3GAT3
GANAB
INTS5
LBHD1
CSKMT
UQCC3
UBXN1
BSCL2
GNG3
HNRNPUL2
TTC9C
ZBTB3
POLR2G
TAF6L
TMEM223
TMEM179B
NXF1
STX5
WDR74
SLC3A2
PLAAT5
PLAAT3
ATL3
RTN3
C11orf95
SPINDOC
MARK2
RCOR2
NAA40
COX8A
OTUB1
MACROD1
TRPT1
NUDT22
DNAJC4
VEGFB
PPP1R14B
PLCB3
GPR137
ESRRA
TRMT112
PRDX5
CCDC88B
NRXN2
RASGRP2
PYGM
SF1
MAP4K2
MEN1
CDC42BPG
EHD1
ATG2A
PPP2R5B
ARL2
SNX15
SAC3D1
CDCA5
ZFPL1
TM7SF2
ZNHIT2
SYVN1
CAPN1
POLA2
CDC42EP2
DPF2
SCYL1
LTBP3
ZNRD2
FAM89B
EHBP1L1
MAP3K11
PCNX3
SIPA1
RELA
KAT5
RNASEH2C
CFL1
MUS81
FIBP
CCDC85B
FOSL1
C11orf68
DRAP1
TSGA10IP
SART1
EIF1AD
BANF1
GAL3ST3
SF3B2
PACS1
KLC2
RAB1B
CNIH2
YIF1A
CD248
BRMS1
B4GAT1
SLC29A2
PELI3
BBS1
ZDHHC24
ACTN3
CCS
RBM14
RBM4B
SPTBN2
C11orf80
RCE1
PC
LRFN4
SYT12
RHOD
KDM2A
GRK2
ANKRD13D
RAD9A
POLD4
PPP1CA
CORO1B
TMEM134
AIP
PITPNM1
CDK2AP2
GSTP1
NDUFV1
NUDT8
UNC93B1
NDUFS8
TCIRG1
CHKA
KMT5B
C11orf24
LRP5
PPP6R3
GAL
TESMIN
MRGPRF
TPCN2
CCND1
LTO1
FGF19
ANO1
FADD
CTTN
SHANK2
DHCR7
AP002495.1
RNF121
IL18BP
NUMA1
LRTOMT
LAMTOR1
ANAPC15
INPPL1
CLPB
ARAP1
STARD10
ATG16L2
FCHSD2
ARHGEF17
RELT
FAM168A
PLEKHB1
RAB6A
COA4
PAAF1
UCP2
C2CD3
PPME1
PGM2L1
KCNE3
POLD3
RNF169
XRRA1
ARRB1
GDPD5
SERPINH1
DGAT2
UVRAG
THAP12
GVQW3
EMSY
TSKU
ACER3
CAPN5
PAK1
CLNS1A
RSF1
AAMDC
INTS4
KCTD14
NDUFC2
ALG8
USP35
GAB2
NARS2
TENM4
PRCP
DDIAS
RAB30
PCF11
ANKRD42
CCDC90B
DLG2
TMEM126B
TMEM126A
CREBZF
SYTL2
PICALM
EED
HIKESHI
PRSS23
FZD4
TMEM135
RAB38
CTSC
NAALAD2
CHORDC1
FAT3
SLC36A4
SMCO4
CEP295
TAF1D
C11orf54
MED17
PANX1
MRE11
ANKRD49
FUT4
AMOTL1
CWC15
SRSF8
SESN3
FAM76B
CEP57
MTMR2
MAML2
CCDC82
JRKL
ARHGAP42
CEP126
YAP1
BIRC2
TMEM123
DCUN1D5
DYNC2H1
PDGFD
GRIA4
MSANTD4
KBTBD3
AASDHPPT
GUCY1A2
CWF19L2
ALKBH8
SLC35F2
RAB39A
CUL5
ACAT1
NPAT
ATM
POGLUT3
EXPH5
DDX10
RDX
ZC3H12C
FDX1
LAYN
SIK2
PPP2R1B
ALG9
FDXACB1
C11orf1
CRYAB
HSPB2
C11orf52
DIXDC1
DLAT
NKAPD1
TIMM8B
PTS
NCAM1
TTC12
ZW10
USP28
HTR3A
C11orf71
RBM7
REXO2
CADM1
BUD13
ZPR1
SIK3
PAFAH1B2
TAGLN
PCSK7
RNF214
BACE1
CEP164
FXYD6
UBE4A
ATP5MG
KMT2A
TMEM25
IFT46
ARCN1
PHLDB1
DDX6
BCL9L
CCDC84
TRAPPC4
SLC37A4
HYOU1
VPS11
HMBS
H2AFX
DPAGT1
HINFP
NLRX1
CBL
MCAM
RNF26
USP2
THY1
NECTIN1
TLCD5
ARHGEF12
TBCEL
SC5D
SORL1
UBASH3B
HSPA8
CLMP
GRAMD1B
ZNF202
VWA5A
TBRG1
SIAE
SPA17
MSANTD2
ROBO3
CCDC15
TMEM218
FEZ1
EI24
STT3A
CHEK1
HYLS1
PUS3
DDX25
CDON
RPUSD4
FAM118B
SRPRA
FOXRED1
DCPS
ST3GAL4
ETS1
ARHGAP32
NFRKB
PRDM10
APLP2
ST14
ZBTB44
ADAMTS8
SNX19
NTM
IGSF9B
JAM3
NCAPD3
VPS26B
THYN1
ACAD8
GLB1L3
GLB1L2
B3GAT1
IQSEC3
KDM5A
CCDC77
B4GALNT3
WNK1
RAD52
ERC1
ADIPOR2
DCP1B
FKBP4
ITFG2
FOXM1
RHNO1
TULP3
TEAD4
TSPAN9
CRACR2A
PARP11
CCND2
TIGAR
C12orf4
RAD51AP1
DYRK4
NDUFA9
GALNT8
CD9
PLEKHG6
TNFRSF1A
SCNN1A
LTBR
VAMP1
NCAPD2
IFFO1
NOP2
CHD4
ING4
ZNF384
PIANP
COPS7A
MLF2
PTMS
LAG3
GPR162
P3H3
GNB3
CDCA3
USP5
SPSB2
ENO2
ATN1
C12orf57
PTPN6
PHB2
EMG1
LPCAT3
C1S
C1R
C1RL
CLSTN3
PEX5
GDF3
NANOG
SLC2A3
FOXJ2
NECAP1
RIMKLB
A2ML1
PHC1
M6PR
KLRG1
CLEC2D
MAGOHB
YBX3
TAS2R14
TAS2R30
SMIM10L1
ETV6
LRP6
MANSC1
BORCS5
DUSP16
CREBL2
GPR19
CDKN1B
HEBP1
FAM234B
GSG1
GRIN2B
ATF7IP
PLBD1
HIST4H4
H2AFJ
WBP11
C12orf60
PTPRO
EPS8
STRAP
DERA
MGST1
LMO3
PLEKHA5
AEBP2
PDE3A
SLCO1A2
PYROXD1
RECQL
GOLT1B
LDHB
CMAS
C2CD5
ETNK1
BCAT1
KRAS
RASSF8
BHLHE41
ITPR2
INTS13
FGFR1OP2
TM7SF3
MED21
STK38L
ARNTL2
PPFIBP1
KLHL42
CCDC91
FAR2
ERGIC2
TMTC1
IPO8
CAPRIN2
TSPAN11
SINHCAF
DENND5B
AMN1
RESF1
BICD1
FGD4
DNM1L
YARS2
PKP2
SYT10
CPNE8
KIF21A
SLC2A13
CNTN1
GXYLT1
PPHLN1
ZCRB1
PRICKLE1
ADAMTS20
TWF1
NELL2
ANO6
ARID2
SCAF11
SLC38A1
SLC38A2
AMIGO2
PCED1B
RPAP3
RAPGEF3
SLC48A1
HDAC7
TMEM106C
COL2A1
SENP1
PFKM
ASB8
ZNF641
KANSL2
CCNT1
ADCY6
CACNB3
DDX23
RND1
FKBP11
ARF3
PRKAG1
KMT2D
RHEBL1
LMBR1L
TUBA1B
TUBA1A
TUBA1C
TROAP
SPATS2
MCRS1
PRPF40B
FMNL3
TMBIM6
NCKAP5L
BCDIN3D
SMARCD1
COX14
CERS5
LIMA1
LARP4
DIP2B
ATF1
METTL7A
SLC11A2
LETMD1
CSRNP2
TFCP2
POU6F1
DAZAP2
SMAGP
GALNT6
SLC4A8
SCN8A
FIGNL2
ACVR1B
GRASP
NR4A1
ATG101
KRT7
KRT8
KRT18
EIF4B
TNS2
SPRYD3
IGFBP6
ZNF740
RARG
MFSD5
ESPL1
PFDN5
C12orf10
AAAS
SP1
MAP3K12
TARBP2
ATF7
ATP5MC2
CALCOCO1
SMUG1
CBX5
COPZ1
ZNF385A
ITGA5
PPP1R1A
METTL7B
ITGA7
BLOC1S1
CD63
GDF11
SARNP
ORMDL2
DNAJC14
PYM1
DGKA
PMEL
CDK2
RAB5B
SUOX
IKZF4
ERBB3
PA2G4
ESYT1
ZC3H10
MYL6B
MYL6
SMARCC2
RNF41
NABP2
ANKRD52
COQ10A
CNPY2
PAN2
STAT2
TIMELESS
SPRYD4
GLS2
BAZ2A
ATP5F1B
PRIM1
HSD17B6
ZBTB39
TAC3
NEMP1
NAB2
STAT6
LRP1
SHMT2
R3HDM2
INHBE
GLI1
MARS
MBD6
DCTN2
KIF5A
PIP4K2C
DTX3
OS9
TSPAN31
CDK4
MARCH9
METTL1
TSFM
CTDSP2
ATP23
LRIG3
USP15
MON2
PPM1H
RXYLT1
SRGAP1
C12orf66
C12orf56
XPOT
TBK1
RASSF3
GNS
TBC1D30
WIF1
LEMD3
HMGA2
LLPH
TMBIM4
HELB
GRIP1
CAND1
DYRK2
MDM1
NUP107
SLC35E3
MDM2
CPSF6
YEATS4
FRS2
CCT2
RAB3IP
MYRFL
CNOT2
KCNMB4
ZFC3H1
THAP2
TMEM19
RAB21
TBC1D15
TRHDE
ATXN7L3B
KRR1
PHLDA1
BBS10
OSBPL8
ZDHHC17
CSRP2
E2F7
SYT1
PAWR
PPP1R12A
LIN7A
ACSS3
CCDC59
METTL25
TMTC2
SLC6A15
NTS
MGAT4C
C12orf29
CEP290
TMTC3
DUSP6
POC1B
GALNT4
ATP2B1
BTG1
EEA1
UBE2N
SOCS2
CRADD
CEP83
TMCC3
NDUFA12
NR2C1
FGD6
VEZT
METAP2
USP44
SNRPF
LTA4H
ELK3
CDK17
NEDD1
TMPO
SLC25A3
IKBIP
APAF1
ANKS1B
UHRF1BP1L
ACTR6
DEPDC4
SCYL2
GAS2L3
UTP20
ARL1
CHPT1
GNPTAB
DRAM1
WASHC3
NUP37
PARPBP
NT5DC3
HSP90B1
TDG
GLT8D2
NFYB
TXNRD1
EID3
CHST11
SLC41A2
C12orf45
ALDH1L2
WASHC4
APPL2
C12orf75
NUAK1
CKAP4
TCP11L2
POLR3B
RIC8B
TMEM263
MTERF2
CRY1
BTBD11
PWP1
PRDM4
FICD
SART3
ISCU
CORO1C
SSH1
USP30
ALKBH2
UNG
ACACB
KCTD10
UBE3B
MMAB
MVK
TRPV4
GLTP
TCHP
GIT2
ANKRD13A
C12orf76
IFT81
ATP2A2
ANAPC7
ARPC3
GPN3
FAM216A
PPTC7
TCTN1
HVCN1
PPP1CC
SH2B3
ATXN2
BRAP
ACAD10
ALDH2
MAPKAPK5
TMEM116
ERP29
NAA25
TRAFD1
HECTD4
PTPN11
DTX1
DDX54
RITA1
IQCD
TPCN1
PLBD2
RBM19
MED13L
RNFT2
HRK
FBXW8
TESC
FBXO21
KSR2
RFC5
WSB2
VSIG10
PEBP1
TAOK3
SUDS3
HSPB8
PRKAB1
CIT
BICDL1
RAB35
GCN1
PXN
MSI1
COX6A1
TRIAP1
GATC
SRSF9
COQ5
RNF10
POP5
CABP1
MLEC
UNC119B
ACADS
SPPL3
C12orf43
P2RX4
CAMKK2
ANAPC5
KDM2B
ORAI1
TMEM120B
RHOF
SETD1B
PSMD9
BCL7A
MLXIP
VPS33A
CLIP1
ZCCHC8
RSRC2
KNTC1
DENR
HIP1R
VPS37B
ABCB9
OGFOD2
ARL6IP4
PITPNM2
MPHOSPH9
C12orf65
CDK2AP1
SBNO1
KMT5A
RILPL2
SNRNP35
RILPL1
TMED2
DDX55
EIF2B1
GTF2H3
TCTN2
ATP6V0A2
ZNF664
RFLNA
NCOR2
SCARB1
UBC
DHX37
BRI3BP
AACS
TMEM132B
SLC15A4
GLT1D1
TMEM132D
STX2
SFSWAP
MMP17
ULK1
PUS1
EP400
DDX51
NOC4L
FBRSL1
POLE
PXMP2
PGAM5
ANKLE2
GOLGA3
CHFR
ZNF605
ZNF26
ZNF84
ZNF140
ZNF891
ZNF10
ZNF268
MPHOSPH8
PSPC1
ZMYM5
ZMYM2
IFT88
EEF1AKMT1
XPO4
LATS2
SAP18
SKA3
ZDHHC20
MICU2
SACS
MIPEP
SPATA13
PARP4
CENPJ
MTMR6
NUP58
ATP8A2
SHISA2
RNF6
CDK8
WASF3
USP12
RASL11A
GTF3A
MTIF3
LNX2
POLR1D
PAN3
FLT1
POMP
SLC7A1
UBL3
KATNAL1
USPL1
HSPH1
B3GLCT
BRCA2
N4BP2L2
PDS5B
RFC3
NBEA
DCLK1
SOHLH2
CCDC169
SPART
RFXAP
ALG5
EXOSC8
SUPT20H
TRPC4
UFM1
FREM2
PROSER1
NHLRC3
LHFPL6
COG6
FOXO1
SLC25A15
ELF1
WBP4
KBTBD6
KBTBD7
MTRF1
NAA16
VWA8
DGKH
AKAP11
TNFSF11
LACC1
TSC22D1
NUFIP1
GPALPP1
GTF2F2
SLC25A30
COG3
ZC3H13
LRCH1
ESD
SUCLA2
NUDT15
MED4
ITM2B
RB1
LPAR6
RCBTB2
FNDC3A
CDADC1
CAB39L
SETDB2
PHF11
RCBTB1
EBPL
KPNA3
SPRYD7
TRIM13
RNASEH2B
C13orf42
FAM124A
INTS6
WDFY2
DHRS12
ALG11
NEK3
VPS36
CKAP2
SUGT1
CNMD
DIAPH3
TDRD3
MZT1
BORA
DIS3
PIBF1
KLF12
TBC1D4
COMMD6
UCHL3
LMO7
KCTD12
CLN5
FBXL3
MYCBP2
SLAIN1
EDNRB
OBI1
RBM26
NDFIP2
SPRY2
SLITRK5
GPC6
TGDS
GPR180
ABCC4
CLDN10
DZIP1
DNAJC3
UGGT2
RAP2A
IPO5
FARP1
STK24
SLC15A1
DOCK9
UBAC2
TM9SF2
CLYBL
ZIC5
ZIC2
PCCA
TMTC4
TPP2
TEX30
POGLUT2
BIVM
ERCC5
EFNB2
ARGLU1
FAM155A
LIG4
ABHD13
IRS2
COL4A1
COL4A2
RAB20
NAXD
CARS2
ING1
ANKRD10
ARHGEF7
TUBGCP3
ATP11A
MCF2L
PCID2
CUL4A
LAMP1
GRTP1
DCUN1D2
TMCO3
TFDP1
GAS6
RASA3
CDC16
UPF3A
CHAMP1
TTC5
CCNB1IP1
PARP2
TEP1
OSGEP
APEX1
PIP4P1
PNP
METTL17
NDRG2
ARHGEF40
ZNF219
HNRNPC
SUPT16H
CHD8
RAB2B
TOX4
METTL3
SALL2
ABHD4
OXA1L
SLC7A7
MMP14
RBM23
PRMT5
HAUS4
AJUBA
C14orf93
PSMB5
CDH24
ACIN1
C14orf119
SLC7A8
HOMEZ
SLC22A17
EFS
NGDN
ZFHX2
AP1G2
JPH4
CARMIL3
PCK2
DCAF11
PSME1
EMC9
RNF31
IRF9
REC8
IPO4
TM9SF1
CHMP4A
MDP1
NEDD8
GMPR2
TINF2
RABGGTA
DHRS1
NOP9
LTB4R
NFATC4
NYNRIN
KHNYN
SDR39U1
STXBP6
NOVA1
PRKD1
G2E3
SCFD1
COCH
STRN3
AP4S1
HECTD1
DTD2
NUBPL
ARHGAP5
EGLN3
SPTSSA
EAPP
SNX6
CFL2
BAZ1A
SRP54
FAM177A1
PPP2R3C
PRORP
NFKBIA
RALGAPA1
BRMS1L
MBIP
SLC25A21
MIPOL1
SEC23A
GEMIN2
TRAPPC6B
PNN
MIA2
FBXO33
C14orf28
KLHL28
TOGARAM1
PRPF39
FKBP3
FANCM
MIS18BP1
LRR1
MGAT2
DNAAF2
POLE2
KLHDC2
NEMF
ARF6
VCPKMT
SOS2
L2HGDH
DMAC2L
CDKL1
MAP4K5
ATL1
SAV1
NIN
PYGL
TRIM9
TMX1
FRMD6
GNG2
RTRAF
NID2
TXNDC16
GPR137C
ERO1A
PSMC6
STYX
GNPNAT1
FERMT2
DDHD1
CDKN3
CNIH1
GMFB
CGRRF1
SAMD4A
WDHD1
SOCS4
MAPK1IP1L
LGALS3
DLGAP5
FBXO34
ATG14
KTN1
PELI2
TMEM260
OTX2
EXOC5
AP5M1
NAA30
ARMH4
ACTR10
PSMA3
ARID4A
TIMM9
KIAA0586
DACT1
DAAM1
L3HYPDH
JKAMP
RTN1
PCNX4
DHRS7
PPM1A
SIX4
MNAT1
TRMT5
SLC38A6
TMEM30B
HIF1A
SNAPC1
SYT16
PPP2R5E
WDR89
SGPP1
SYNE2
MTHFD1
ZBTB25
AKAP5
ZBTB1
HSPA2
PLEKHG3
CHURC1
RAB15
FNTB
MAX
FUT8
GPHN
MPP5
ATP6V1D
EIF2S1
PLEKHH1
PIGH
ARG2
VTI1B
RDH11
ZFYVE26
RAD51B
ZFP36L1
ACTN1
DCAF5
EXD2
GALNT16
ERH
SLC39A9
CCDC177
SUSD6
SRSF5
COX16
SYNJ2BP
MED6
TTC9
MAP3K9
PCNX1
SIPA1L1
DCAF4
ZFYVE1
RBM25
PSEN1
PAPLN
NUMB
RIOX1
DNAL1
PNMA1
ELMSAN1
PTGR2
ZNF410
COQ6
ENTPD5
ALDH6A1
LIN52
ABCD4
VRTN
SYNDIG1L
NPC2
ISCA2
AREL1
FCF1
DLST
PGF
EIF2B2
MLH3
ACYP1
ZC2HC1C
NEK9
TMED10
FOS
JDP2
FLVCR2
TTLL5
ERG28
IFT43
GPATCH2L
VASH1
ANGEL1
IRF2BPL
CIPC
TMEM63C
ZDHHC22
POMT2
GSTZ1
TMED8
SAMD15
VIPAS39
AHSA1
SPTLC2
ALKBH1
SLIRP
SNW1
NRXN3
CEP128
GTF2A1
STON2
SEL1L
FLRT2
GALC
SPATA7
PTPN21
ZC3H14
TTC8
FOXN3
EFCAB11
TDP1
PSMC1
CALM1
TTC7B
DGLUCY
CCDC88C
PPP4R3A
TC2N
TRIP11
ATXN3
NDUFB1
CPSF2
RIN3
LGMN
GOLGA5
CHGA
ITPK1
MOAP1
TMEM251
GON7
UBR7
BTBD7
UNC79
PRIMA1
OTUB2
DDX24
IFI27L1
IFI27L2
PPP4R4
DICER1
GLRX5
TCL1B
TUNAR
C14orf132
ATG2B
GSKIP
AK7
PAPOLA
VRK1
BCL11B
SETD3
CCNK
CCDC85C
HHIPL1
EML1
EVL
YY1
SLC25A29
WARS
WDR25
PPP2R5C
DYNC1H1
HSP90AA1
WDR20
ZNF839
CINP
TECPR2
ANKRD9
RCOR1
TRAF3
CDC42BPB
TNFAIP2
EIF5
MARK3
CKB
TRMT61A
BAG5
KLC1
COA8
XRCC3
ZFYVE21
PPP1R13B
ATP5MPL
KIF26A
INF2
ADSSL1
SIVA1
AKT1
CEP170B
CDCA4
JAG2
NUDT14
BRF1
BTBD6
PACS2
MTA1
CRIP1
TEDC1
NIPA1
NIPA2
CYFIP1
TUBGCP5
MAGEL2
NDN
SNRPN
UBE3A
GABRB3
GABRA5
HERC2
APBA2
FAM189A1
NSMCE3
TJP1
AC091057.6
ARHGAP11B
FAN1
MTMR10
KLF13
OTUD7A
CHRNA7
ARHGAP11A
SCG5
AVEN
EMC7
PGBD4
KATNBL1
EMC4
SLC12A6
NOP10
LPCAT4
GOLGA8B
ACTC1
AQR
ZNF770
DPH6
C15orf41
SPRED1
FAM98B
RASGRP1
THBS1
GPR176
EIF2AK4
BMF
BUB1B
PAK6
PLCB2
INAFM2
KNSTRN
IVD
BAHD1
CHST14
CCDC32
RPUSD2
KNL1
RAD51
RMDN3
GCHFR
ZFYVE19
SPINT1
RHOV
VPS18
CHAC1
INO80
CHP1
OIP5
NUSAP1
NDUFAF1
RTF1
RPAP1
TYRO3
MGA
MAPKBP1
PLA2G4B
EHD4
VPS39
TMEM87A
GANC
ZNF106
SNAP23
LRRC57
HAUS2
TTBK2
CCNDBP1
LCMT2
ADAL
ZSCAN29
TUBGCP4
TP53BP1
PPIP5K1
PDIA3
ELL3
SERF2
MFAP1
WDR76
FRMD5
CASC4
CTDSPL2
EIF3J
SPG11
B2M
SORD
SHF
GATM
SPATA5L1
BLOC1S6
SEMA6D
MYEF2
DUT
CEP152
EID1
SECISBP2L
COPS2
GALK2
FAM227B
DTWD1
SLC27A2
GABPB1
USP8
TRPM7
SPPL2A
AP4E1
DMXL2
SCG3
LYSMD2
TMOD2
TMOD3
LEO1
MAPK6
GNB5
MYO5C
MYO5A
ARPP19
FAM214A
ONECUT1
RSL24D1
RAB27A
PIGBOS1
PIGB
CCPG1
DNAAF4
PRTG
NEDD4
RFX7
TEX9
MNS1
ZNF280D
TCF12
CGNL1
ADAM10
MINDY2
RNF111
SLTM
CCNB2
MYO1E
GTF2A2
BNIP2
ANXA2
ICE2
RORA
VPS13C
TLN2
TPM1
RAB8B
APH1B
CA12
USP3
HERC1
CIAO2A
SNX1
PPIB
CSNK1G1
PCLAF
TRIP4
ZNF609
OAZ2
RBPMS2
PIF1
PLEKHO2
ANKDD1A
SPG21
MTFMT
RASL12
PDCD7
CLPX
PARP16
IGDCC3
IGDCC4
DPP8
HACD3
INTS14
DENND4A
RAB11A
DIS3L
TIPIN
MAP2K1
SNAPC5
ZWILCH
SMAD3
AAGAB
C15orf61
MAP2K5
PIAS1
CALML4
CLN6
FEM1B
CORO2B
ANP32A
GLCE
PAQR5
KIF23
UACA
LRRC49
THAP10
THSD4
MYO9A
SENP8
PKM
PARP6
HEXA
ARIH1
BBS4
ADPGK
NEO1
NPTN
CD276
INSYN1
LOXL1
PML
STRA6
SEMA7A
UBL7
ARID3B
CLK3
EDC3
CSK
ULK3
SCAMP2
MPI
FAM219B
RPP25
SCAMP5
PPCDC
C15orf39
COMMD4
SIN3A
PTPN9
SNUPN
IMP3
SNX33
CSPG4
UBE2Q2
FBXO22
TMEM266
ETFA
SCAPER
RCN2
TSPAN3
PEAK1
HMG20A
LINGO1
TBC1D2B
CIB2
IDH3A
ACSBG1
DNAJA4
WDR61
CRABP1
IREB2
HYKK
PSMA4
CHRNA5
CHRNA3
ADAMTS7
MORF4L1
CTSH
TMED3
MTHFS
AC015871.1
FAH
ARNT2
ABHD17C
MESD
TLNRD1
STARD5
MEX3B
EFL1
SAXO2
CPEB1
AP3B2
WHAMM
HOMER2
RAMAC
C15orf40
BTBD1
HDGFL3
SH3GL3
ZSCAN2
WDR73
NMB
SEC11A
ZNF592
ALPK3
PDE8A
AKAP13
KLHL25
NTRK3
AEN
ISG20
HAPLN3
MFGE8
ABHD2
FANCI
POLG
TICRR
KIF7
PEX11A
AP3S2
ARPIN
ZNF710
IDH2
SEMA4B
CIB1
GDPGP1
NGRN
IQGAP1
BLM
FURIN
FES
MAN2A2
HDDC3
UNC45A
RCCD1
PRC1
VPS33B
SLCO3A1
ST8SIA2
FAM174B
CHD2
RGMA
MCTP2
ARRDC4
IGF1R
TTC23
LRRC28
MEF2A
LYSMD4
LINS1
ASB7
ALDH1A3
CHSY1
SELENOS
SNRPA1
PCSK6
TM2D3
TARSL2
POLR3K
SNRNP25
RHBDF1
MPG
NPRL3
HBQ1
LUC7L
FAM234A
AXIN1
TMEM8A
NME4
RAB11FIP3
CAPN15
PIGQ
RAB40C
WFIKKN1
METTL26
WDR90
RHOT2
STUB1
JMJD8
WDR24
FBXL16
METRN
ANTKMT
HAGHL
CIAO3
RPUSD1
CHTF18
LMF1
CACNA1H
UBE2I
TSR3
GNPTG
CLCN7
TELO2
IFT140
CRAMP1
JPT2
MAPK8IP3
NME3
SPSB3
NUBP2
HAGH
FAHD1
MSRB1
NDUFB10
TBL3
GFER
SYNGR3
ZNF598
SLC9A3R2
NTHL1
TSC2
PKD1
TRAF7
CASKIN1
MLST8
PGP
E4F1
ECI1
RNPS1
ABCA3
CCNF
TEDC2
TBC1D24
ATP6V0C
AMDHD2
PDPK1
KCTD5
PRSS27
SRRM2
FLYWCH2
KREMEN2
PKMYT1
PAQR4
CLDN6
TNFRSF12A
HCFC1R1
THOC6
BICDL2
MMP25
ZSCAN10
ZNF205
ZNF213
ZNF200
ZNF263
TIGD7
ZNF75A
ZSCAN32
ZNF174
NAA60
CLUAP1
SLX4
TRAP1
CREBBP
ADCY9
TFAP4
GLIS2
PAM16
CORO7
VASN
DNAJA3
NMRAL1
HMOX2
CDIP1
UBALD1
MGRN1
NUDT16L1
ANKS3
ROGDI
GLYR1
UBN1
PPL
SEC14L5
NAGPA
ALG1
C16orf89
EEF2KMT
RBFOX1
METTL22
ABAT
TMEM186
PMM2
CARHSP1
USP7
C16orf72
ATF7IP2
EMP2
NUBP1
TVP23A
DEXI
CLEC16A
RMI2
SOCS1
AC099489.1
LITAF
SNN
TXNDC11
ZC3H7A
RSL1D1
GSPT1
SNX29
CPPED1
SHISA9
ERCC4
MRTFB
PARN
BFAR
NOMO1
NTAN1
MPV17L
BMERB1
MARF1
NDE1
FOPNL
ABCC1
XYLT1
ARL6IP1
SMG1
TMC7
COQ7
ITPRIPL2
SYT17
GDE1
CCP110
VPS35L
KNOP1
IQCK
GPRC5B
THUMPD1
ERI2
REXO5
DCUN1D3
LYRM1
TMEM159
CRYM
METTL9
UQCRC2
MOSMO
EEF2K
POLR3E
CDR2
HS3ST2
USP31
COG7
GGA2
EARS2
UBFD1
NDUFAB1
PALB2
PLK1
PRKCB
RBBP6
TNRC6A
ARHGAP17
LCMT1
ZKSCAN2
HS3ST4
NSMCE1
GTF3C1
KIAA0556
XPO6
SBK1
CLN3
SGF29
ATXN2L
TUFM
SH2B1
RABEP2
NFATC2IP
SPNS1
SPN
QPRT
KIF22
PAGR1
CDIPT
SEZ6L2
ASPHD1
KCTD13
TMEM219
TAOK2
HIRIP3
INO80E
DOC2A
TLCD3B
PPP4C
YPEL3
MAPK3
CORO1A
CD2BP2
TBC1D10B
ZNF48
ZNF771
DCTPP1
SEPHS2
ZNF768
ZNF785
ZNF689
PRR14
FBRS
SRCAP
PHKG2
RNF40
ZNF629
BCL7C
FBXL19
ORAI3
SETD1A
STX1B
STX4
ZNF668
ZNF646
VKORC1
BCKDK
KAT8
PRSS8
FUS
PYCARD
TGFB1I1
C16orf58
ZNF267
SHCBP1
ORC6
C16orf87
GPT2
DNAJA2
NETO2
ITFG1
PHKB
LONP2
SIAH1
N4BP1
CBLN1
ZNF423
CNEP1R1
HEATR3
TENT4B
BRD7
NKD1
CYLD
SALL1
TOX3
AC007906.2
CHD9
RBL2
AKTIP
RPGRIP1L
FTO
MMP2
LPCAT2
GNAO1
AMFR
NUDT21
OGFOD1
BBS2
MT2A
MT1X
NUP93
HERPUD1
CPNE2
FAM192A
RSPRY1
ARL2BP
CIAPIN1
COQ9
POLR2C
DOK4
CCDC102A
ADGRG5
ADGRG1
KATNB1
USB1
MMP15
CFAP20
CSNK2A2
CCDC113
GINS3
NDRG4
SETD6
CNOT1
SLC38A7
GOT2
CDH8
CDH11
TK2
CKLF
CMTM3
CMTM4
DYNC1LI2
NAE1
CA7
PDP2
CIAO2B
CES2
CBFB
C16orf70
KIAA0895L
E2F4
TMEM208
FHOD1
SLC9A5
PLEKHG4
HSD11B2
ATP6V0D1
RIPOR1
CTCF
CARMIL2
ACD
PARD6A
ENKD1
GFOD2
RANBP10
CENPT
THAP11
NUTF2
EDC4
NRN1L
PSKH1
PSMB10
SLC12A4
DPEP3
DUS2
DDX28
NFATC3
ESRP2
PLA2G15
SLC7A6
SLC7A6OS
PRMT7
ZFP90
CDH3
CDH1
TANGO6
HAS3
UTP4
SNTB2
VPS4A
COG8
NIP7
TERF2
CYB5B
NFAT5
NQO1
NOB1
WWP2
PDPR
EXOSC6
AARS
DDX19B
DDX19A
ST3GAL2
COG4
SF3B3
IL34
MTSS2
VAC14
HYDIN
CMTR2
CALB2
ZNF23
ZNF19
CHST4
MARVELD3
PHLPP2
AP1G1
ATXN1L
ZNF821
DHODH
TXNL4B
DHX38
PSMD7
GLG1
RFWD3
WDR59
ZNRF1
ZFP1
BCAR1
CFDP1
TMEM170A
CHST6
TMEM231
ADAT1
KARS
TERF2IP
MON1B
NUDT7
VAT1L
WWOX
CDYL2
CMC2
CENPN
ATMIN
C16orf46
GAN
CMIP
PLCG2
MPHOSPH6
CDH13
HSBP1
MLYCD
MBTPS1
HSDL1
TAF1C
ATP2C2
MEAK7
COTL1
KLHL36
USP10
ZDHHC7
KIAA0513
GSE1
GINS2
EMC8
COX4I1
MTHFSD
C16orf95
FBXO31
MAP1LC3B
ZCCHC14
JPH3
KLHDC4
SLC7A5
BANP
ZC3H18
CYBA
MVD
SNAI3
RNF166
CTU2
CDT1
APRT
GALNS
TRAPPC2L
ACSF3
SLC22A31
ZNF778
ANKRD11
SPG7
CPNE7
DPEP1
CHMP1A
SPATA33
SPATA2L
ZNF276
FANCA
SPIRE2
TCF25
TUBB3
CENPBD1
DBNDD1
GAS8
C17orf97
RFLNB
VPS53
TLCD3A
GEMIN4
MRM3
NXN
TIMM22
ABR
YWHAE
CRK
MYO1C
INPP5K
PITPNA
SLC43A2
RILP
PRPF8
WDR81
SERPINF1
SMYD4
RPA1
SMG6
TSR1
SGSM2
METTL16
PAFAH1B1
CLUH
RAP1GAP2
TRPV1
SHPK
CTNS
TAX1BP3
EMC6
P2RX5
HASPIN
NCBP3
CAMKK1
ATP2A3
ZZEF1
CYB5D2
ANKFY1
UBE2G1
MYBBP1A
SMTNL2
PELP1
ARRB2
MED11
CXCL16
PSMB6
PLD2
MINK1
SLC25A11
RNF167
PFN1
ENO3
SPAG7
CAMTA2
KIF1C
ZNF232
ZNF594
RABEP1
NUP88
RPAIN
C1QBP
DHX33
DERL2
MIS12
WSCD1
PIMREG
PITPNM3
KIAA0753
TXNDC17
MED31
RNASEK
C17orf49
SLC16A13
DLG4
ACADVL
DVL2
PHF23
GABARAP
ELP5
CLDN7
SLC2A4
GPS2
NEURL4
ACAP1
TNK1
PLSCR3
TMEM256
NLGN2
TMEM102
FGF11
CHRNB1
ZBTB4
POLR2A
TNFSF12
TNFSF13
SENP3
EIF4A1
MPDU1
FXR2
SAT2
ATP1B2
TP53
WRAP53
EFNB3
KDM6B
NAA38
CYB5D1
CHD3
TRAPPC1
CNTROB
PER1
VAMP2
TMEM107
BORCS6
AURKB
CTC1
PFAS
SLC25A35
RANGRF
KRBA2
NDEL1
MYH10
MFSD6L
PIK3R5
STX8
USP43
GAS7
SCO1
ADPRM
TMEM220
ZNF18
MAP2K4
ARHGAP44
ELAC2
COX10
HS3ST3B1
CDRT4
TVP23C
TRIM16
ZNF286A
ADORA2B
ZSWIM7
TTC19
NCOR1
PIGL
CENPV
UBB
ZNF287
MPRIP
FLCN
COPS3
NT5M
MED9
PEMT
SREBF1
TOM1L2
GID4
DRG2
ALKBH5
LLGL1
FLII
TOP3A
SMCR8
SHMT1
PRPSAP2
FAM83G
EPN2
B9D1
MAPK7
MFAP4
ALDH3A2
ULK2
AKAP10
SPECC1
USP22
DHRS7B
TMEM11
NATD1
MAP2K3
WSB1
KSR1
LYRM9
NLK
TMEM97
IFT20
TNFAIP1
POLDIP2
TMEM199
SARM1
SLC46A1
UNC119
PIGS
ALDOC
SPAG5
KIAA0100
SDF2
SUPT6H
PROCA1
RAB34
TLCD1
NEK8
TRAF4
FAM222B
ERAL1
FLOT2
DHRS13
PHF12
PIPOX
MYO18A
NUFIP2
TAOK1
GIT1
ANKRD13B
BLMH
CPD
GOSR1
CRLF3
ATAD5
TEFM
NF1
RAB11FIP4
COPRS
UTP6
SUZ12
LRRC37B
RHOT1
RHBDL3
C17orf75
ZNF207
PSMD11
CDK5R1
MYO1D
TMEM98
ZNF830
LIG3
RFFL
RAD51D
NLE1
SLFN5
SLFN12
SLFN13
PEX12
AP2B1
TAF15
RASL10B
RDM1
MYO19
PIGW
GGNBP2
DHRS11
MRM1
AATF
ACACA
TADA2A
DUSP14
SYNRG
DDX52
SOCS7
ARHGAP23
MLLT6
CISD3
PCGF2
PSMB3
PIP4K2B
CWC25
LASP1
LINC00672
CACNB1
STAC2
FBXL20
MED1
CDK12
PPP1R1B
STARD3
PNMT
PGAP3
ERBB2
MIEN1
GRB7
GSDMB
ORMDL3
PSMD3
MED24
THRA
NR1D1
MSL1
CASC3
WIPF2
CDC6
RARA
TOP2A
IGFBP4
SMARCE1
TMEM99
EIF1
JUP
P3H4
FKBP10
NT5C3B
ACLY
TTC25
CNP
DNAJC7
NKIRAS2
RAB5C
STAT5B
STAT3
CAVIN1
ATP6V0A1
NAGLU
COASY
MLX
PSMC3IP
RETREG3
TUBG1
TUBG2
PLEKHH3
EZH1
RAMP2
VPS25
COA3
BECN1
PSME3
AARSD1
PTGES3L
RUNDC1
IFI35
VAT1
RND2
BRCA1
NBR1
TMEM106A
ARL4D
DHX8
ETV4
DUSP3
MPP3
MPP2
NAGS
TMEM101
LSM12
G6PC3
HDAC5
C17orf53
TMUB2
ATXN7L3
UBTF
RUNDC3A
SLC25A39
GRN
FAM171A2
GPATCH8
FZD2
MEIOC
CCDC43
DBF4B
GJC1
EFTUD2
KIF18B
C1QL1
DCAKD
NMT1
ACBD4
HEXIM1
HEXIM2
FMNL1
ARHGAP27
KANSL1
NSF
GOSR2
CDC27
NPEPPS
KPNB1
TBKBP1
SCRN2
SP2
PNPO
CDK5RAP3
NFE2L1
CBX1
SNX11
TTLL6
CALCOCO2
ATP5MC1
UBE2Z
SNF8
IGF2BP1
ZNF652
PHB
NGFR
NXPH3
SPOP
SLC35B1
FAM117A
KAT7
ITGA3
PDK2
SAMD14
PPP1R9B
COL1A1
TMEM92
XYLT2
EME1
LRRC59
ACSF2
RSAD1
SPATA20
CACNA1G
ANKRD40
LUC7L3
TOB1
SPAG9
NME1
NME2
MBTD1
UTP18
TOM1L1
COX11
STXBP4
MMD
PCTP
DGKE
TRIM25
COIL
SCPEP1
AKAP1
MSI2
CUEDC1
VEZF1
SRSF1
DYNLL2
MKS1
TSPOAP1
SUPT4H1
MTMR4
SEPTIN4
RAD51C
PPM1E
TRIM37
SKA2
PRR11
SMG8
GDPD1
YPEL2
DHX40
CLTC
PTRH2
VMP1
TUBD1
RNFT1
HEATR6
USP32
APPBP2
PPM1D
BCAS3
BRIP1
INTS2
MED13
METTL2A
TLK2
MRC2
TANC2
CYB561
ACE
DCAF7
TACO1
MAP3K3
STRADA
LIMD2
CCDC47
DDX42
FTSJ3
PSMC5
SMARCD2
SCN4A
ERN1
TEX2
POLG2
DDX5
CEP95
SMURF2
GNA13
PRKCA
CACNG5
CACNG4
HELZ
PSMD12
PITPNC1
NOL11
BPTF
C17orf58
KPNA2
AMZ2
ARSG
WIPI1
PRKAR1A
ABCA5
MAP2K6
SLC39A11
COG1
C17orf80
SDK2
TTYH2
BTBD17
GPRC5C
RAB37
SLC9A3R1
NAT9
TMEM104
FDXR
HID1
CDR2L
KCTD2
ATP5PD
SLC16A5
ARMC7
NT5C
JPT1
SUMO2
NUP85
GGA3
MIF4GD
SLC25A19
GRB2
TMEM94
CASKIN2
TSEN54
LLGL2
RECQL5
SAP30BP
ITGB4
GALK1
UNK
WBP2
TRIM47
TRIM65
FBF1
ACOX1
TEN1
CDK3
SRP68
EXOC7
FOXJ1
RNF157
QRICH2
PRPSAP1
UBE2O
RHBDF2
CYGB
MXRA7
JMJD6
METTL23
SRSF2
MFSD11
MGAT5B
SEC14L1
SEPTIN9
TNRC6C
TMC6
SYNGR2
TK1
BIRC5
SOCS3
PGS1
CYTH1
USP36
TIMP2
LGALS3BP
CANT1
ENGASE
RBFOX3
CBX2
CBX4
TBC1D16
CCDC40
GAA
EIF4A3
SGSH
SLC26A11
RNF213
ENDOV
RPTOR
CHMP6
BAIAP2
CEP131
TEPSIN
NDUFAF8
SLC38A10
ACTG1
FAAP100
NPLOC4
PDE6G
OXLD1
CCDC137
ARL16
HGS
SLC25A10
MCRIP1
ARHGDIA
ALYREF
ANAPC11
PCYT2
SIRT7
MAFG
PYCR1
ASPSCR1
CENPX
LRRC45
RAC3
DCXR
RFNG
GPS1
DUS1L
FASN
CCDC57
SLC16A3
CSNK1D
OGFOD3
HEXD
CYBC1
NARF
FOXK2
WDR45B
FN3KRP
FN3K
TBCD
METRNL
USP14
THOC1
COLEC12
TYMS
ENOSF1
YES1
METTL4
NDC80
SMCHD1
EMILIN2
LPIN2
MYL12A
MYL12B
TGIF1
DLGAP1
ZBTB14
EPB41L3
TMEM200C
ARHGAP28
LAMA1
RAB12
MTCL1
NDUFV2
ANKRD12
TWSG1
RALBP1
PPP4R1
RAB31
VAPA
APCDD1
NAPG
PIEZO2
GNAL
CHMP1B
MPPE1
IMPA2
TUBB6
AFG3L2
PRELID3A
SPIRE1
PSMG2
CEP76
PTPN2
SEH1L
CEP192
FAM210A
RNMT
ZNF519
ROCK1
ESCO1
ABHD3
MIB1
RBBP8
CABLES1
TMEM241
RIOK3
RMC1
NPC1
ANKRD29
TTC39C
CABYR
OSBPL1A
IMPACT
ZNF521
SS18
TAF4B
KCTD1
CHST9
CDH2
DSC3
DSC2
DSG2
B4GALT6
TRAPPC8
RNF125
RNF138
GAREM1
DTNA
MAPRE2
ZNF397
ZSCAN30
ZNF24
INO80C
GALNT1
C18orf21
RPRD1A
SLC39A6
ELP2
MOCOS
FHOD3
TPGS2
KIAA1328
PIK3C3
SYT4
SETBP1
EPG5
ATP5F1A
HAUS1
C18orf25
RNF165
ST8SIA5
PIAS2
KATNAL2
HDHD2
IER3IP1
SMAD2
CTIF
SMAD7
DYM
C18orf32
LIPG
ACAA2
MYO5B
CXXC1
SKA1
MAPK4
MRO
ME2
ELAC1
SMAD4
MEX3C
MBD2
POLI
C18orf54
TCF4
TXNL1
ST8SIA3
ONECUT2
FECH
NARS
ATP8B1
NEDD4L
MALT1
ZNF532
SEC11C
LMAN1
CCBE1
PMAIP1
PIGN
RELCH
TNFRSF11A
ZCCHC2
PHLPP1
KDSR
VPS4B
CDH7
DSEL
TMX3
RTTN
SOCS6
NETO1
TIMM21
CYB5A
CNDP2
ZNF407
ZADH2
ZNF516
ZNF236
SALL3
ATP9B
NFATC1
CTDP1
SLC66A2
TXNL4A
RBFA
ADNP2
PARD6G
PLPP2
MIER2
TPGS1
CDC34
BSG
HCN2
RNF126
FSTL3
PALM
PTBP1
PLPPR3
MED16
R3HDM4
ARID3A
WDR18
TMEM259
CNN2
ABCA7
ARHGAP45
POLR2E
GPX4
SBNO2
STK11
ATP5F1D
MIDN
CIRBP
FAM174C
PWWP3A
NDUFS7
GAMT
DAZAP1
C19orf25
REEP6
MEX3D
MBD3
UQCR11
TCF3
REXO1
KLF16
SCAMP4
CSNK1G2
BTBD2
MKNK2
MOB3A
AP3D1
DOT1L
PLEKHJ1
SF3A2
AMH
OAZ1
LSM7
SPPL2B
TIMM13
LMNB2
DIRAS1
SLC39A3
SGTA
THOP1
ZNF554
ZNF555
ZNF57
ZNF77
TLE2
TLE5
GNA11
NCLN
CELF5
SMIM24
DOHH
FZR1
MFSD12
HMG20B
CACTIN
PIP5K1C
TJP3
MATK
ATCAY
NMRK2
DAPK3
EEF2
PIAS4
ZBTB7A
MAP2K2
SIRT6
ANKRD24
YJU2
FSD1
STAP2
MPND
SH3GL1
CHAF1A
UBXN6
HDGFL2
SEMA6B
MYDGF
DPP9
FEM1A
PLIN3
UHRF1
KDM4B
PTPRS
SAFB2
SAFB
MICOS13
LONP1
DUS3L
NDUFA11
RANBP3
MLLT1
CLPP
ALKBH7
GTF2F1
KHSRP
SLC25A23
CRB3
DENND1C
C3
GPR108
TRIP10
SH2D3A
ZNF557
INSR
ARHGEF18
ZNF358
MCOLN1
PNPLA6
CAMSAP3
XAB2
PET100
STXBP2
TRAPPC5
EVI5L
PRR36
LRRC8E
MAP2K7
SNAPC2
CTXN1
TIMM44
ELAVL1
FBN3
CERS4
CD320
NDUFA7
RAB11B
MARCH2
HNRNPM
ZNF414
MYO1F
ADAMTS10
ZNF558
ZNF317
ZNF559
ZNF177
ZNF266
ZNF426
ZNF121
ZNF561
ZNF562
FBXL12
UBL5
PIN1
OLFM2
COL5A3
SHFL
PPAN
P2RY11
EIF3G
DNMT1
S1PR2
ZGLP1
FDX2
RAVER1
ICAM3
TYK2
CDC37
KEAP1
S1PR5
ATG4D
KRI1
CDKN2D
AP1M2
SLC44A2
ILF3
QTRT1
DNM2
TMED1
CARM1
YIPF2
TIMM29
SMARCA4
LDLR
SPC24
KANK2
DOCK6
TMEM205
PLPPR2
SWSAP1
RGL3
CCDC151
PRKCSH
ZNF653
ECSIT
CNN1
ELOF1
ZNF627
ACP5
ZNF823
ZNF69
ZNF700
ZNF878
ZNF844
ZNF20
ZNF625
ZNF136
ZNF44
ZNF563
ZNF442
ZNF709
ZNF490
ZNF791
MAN2B1
WDR83
WDR83OS
DHPS
FBXW9
TNPO2
TRIR
GET3
BEST2
HOOK2
JUNB
PRDX2
RNASEH2A
MAST1
DNASE2
GCDH
FARSA
CALR
RAD23A
GADD45GIP1
TRMT1
NACC1
STX10
IER2
CCDC130
MRI1
C19orf53
ZSWIM4
CC2D1A
DCAF15
RFX1
IL27RA
PALM3
SAMD1
PRKACA
ASF1B
ADGRL1
ADGRE5
DDX39A
PKN1
GIPC1
DNAJB1
NDUFB7
ZNF333
ADGRE2
SLC1A6
SYDE1
ILVBL
NOTCH3
EPHX3
BRD4
AKAP8
AKAP8L
WIZ
TPM4
RAB8A
FAM32A
AP1M1
EPS15L1
C19orf44
CHERP
SLC35E1
MED26
SMIM7
TMEM38A
SIN3B
HAUS8
MYO9B
OCEL1
NR2F6
BABAM1
ANKLE1
ABHD8
DDA1
ANO8
GTPBP3
BST2
MVB12A
PGLS
COLGALT1
UNC13A
MAP1S
FCHO1
CCDC124
KCNN1
MAST3
PIK3R2
IFI30
MPV17L2
RAB3A
JUND
LSM4
PGPEP1
SSBP4
ISYNA1
ELL
FKBP8
KXD1
UBA52
CRLF1
TMEM59L
KLHL26
CRTC1
UPF1
COPE
DDX49
HOMER3
SUGP2
ARMC6
SLC25A42
TMEM161A
MEF2B
BORCS8
RFXANK
NR2C2AP
NCAN
HAPLN4
SUGP1
GATAD2A
NDUFA13
CILP2
PBX4
LPAR2
ATP13A1
ZNF101
ZNF14
ZNF253
ZNF682
ZNF90
ZNF486
ZNF737
ZNF626
ZNF66
ZNF85
ZNF430
ZNF714
ZNF708
ZNF738
ZNF493
ZNF100
ZNF43
ZNF730
ZNF724
ZNF91
ZNF675
ZNF681
ZNF254
UQCRFS1
POP4
PLEKHF1
C19orf12
CCNE1
URI1
TSHZ3
ZNF507
DPY19L3
PDCD5
ANKRD27
NUDT19
CEP89
FAAP24
RHPN2
GPATCH1
LRP3
SLC7A10
CEBPG
PEPD
CHST8
KCTD15
LSM14A
KIAA0355
GPI
PDCD2L
UBA2
WTIP
ZNF181
ZNF599
ZNF30
ZNF792
GRAMD1A
HPN
FXYD1
FXYD7
FXYD5
LSR
USF2
DMKN
TMEM147
HAUS5
RBM42
COX6B1
KMT2B
IGFLR1
U2AF1L4
PSENEN
LIN37
PROSER3
KIRREL2
APLP1
NFKBID
LRFN3
SDHAF1
SYNE4
ALKBH6
CLIP3
WDR62
POLR2I
TBCB
CAPNS1
ZNF565
ZNF146
ZFP14
ZFP82
ZNF566
ZNF260
ZNF529
ZNF382
ZNF567
ZNF850
ZNF790
ZNF345
ZNF568
ZNF420
ZNF585A
ZNF585B
ZNF383
ZNF875
ZNF527
ZNF569
ZNF570
ZNF793
ZNF571
ZFP30
ZNF607
ZNF573
SIPA1L3
DPF1
SPINT2
PPP1R14A
YIF1B
C19orf33
KCNK6
SPRED3
RYR1
ACTN4
ECH1
HNRNPL
RINL
SIRT2
NFKBIB
SARS2
FBXO17
FBXO27
PAK4
NCCRP1
LRFN1
SAMD4B
PAF1
MED29
PLEKHG2
SUPT5H
TIMM50
DLL3
SELENOV
EID2
DYRK1B
FBL
FCGBP
PSMC4
ZNF780B
ZNF780A
MAP3K10
AKT2
C19orf47
PLD3
PRX
SERTAD1
SERTAD3
BLVRB
SHKBP1
LTBP4
NUMBL
COQ8B
ITPKC
C19orf54
SNRPA
RAB4B
EGLN2
CYP2S1
AXL
HNRNPUL1
CCDC97
B9D2
EXOSC5
BCKDHA
DMAC2
ARHGEF1
RABAC1
ATP1A3
GRIK5
ZNF574
POU2F2
DEDD2
ZNF526
GSK3A
ERF
CIC
PAFAH1B3
MEGF8
CNFN
LIPE
ETHE1
XRCC1
PINLYP
IRGQ
ZNF576
ZNF428
CADM4
PLAUR
SMG9
ZNF283
ZNF404
ZNF45
ZNF221
ZNF155
ZNF230
ZNF223
ZNF284
ZNF224
ZNF225
ZNF234
ZNF226
ZNF227
ZNF235
ZNF285
ZNF229
ZNF180
PVR
BCL3
CBLC
BCAM
NECTIN2
TOMM40
APOE
CLPTM1
RELB
CLASRP
ZNF296
GEMIN7
MARK4
PPP1R37
TRAPPC6A
ERCC2
PPP1R13L
CD3EAP
ERCC1
RTN2
VASP
OPA3
EML2
SNRPD2
QPCTL
FBXO46
SIX5
DMWD
SYMPK
FOXA3
IRF2BP1
MYPOP
NOVA2
CCDC61
HIF3A
PPP5C
CCDC8
CALM3
PRKD2
STRN4
FKRP
SLC1A5
AP2S1
ARHGAP35
TMEM160
ZC3H4
SAE1
BBC3
CCDC9
INAFM1
DHX34
SLC8A2
KPTN
NAPA
BICRA
EHD2
NOP53
SELENOW
LIG1
ZSWIM9
ZNF114
EMP3
TMEM143
KDELR1
GRIN2D
GRWD1
CYTH2
LMTK3
FAM83E
SPHK2
DBP
CA11
MAMSTR
RASIP1
FUT1
BCAT2
HSD17B14
PLEKHA4
PPP1R15A
NUCB1
BAX
FTL
GYS1
RUVBL2
SNRNP70
LIN7B
PPFIA3
TRPM4
TEAD2
PIH1D1
ALDH16A1
FCGRT
RCN3
NOSIP
PRRG2
PRR12
RRAS
SCAF1
IRF3
BCL2L12
PRMT1
CPT1C
AP2A1
FUZ
MED25
PTOV1
AKT1S1
TBC1D17
NUP62
ATF5
VRK3
ZNF473
IZUMO2
MYH14
KCNC3
NR1H2
POLD1
SPIB
MYBPC2
EMC10
JOSD2
LRRC4B
SYT3
C19orf81
C19orf48
KLK5
KLK8
KLK10
KLK13
IGLON5
VSIG10L
ETFB
CLDND2
ZNF175
SPACA6
ZNF577
ZNF649
ZNF613
ZNF350
ZNF614
ZNF432
ZNF841
ZNF836
PPP2R1A
ZNF766
ZNF480
ZNF610
ZNF578
ZNF808
ZNF83
ZNF600
ZNF28
ZNF468
ZNF320
ZNF888
ZNF160
ZNF415
ZNF347
ZNF677
ZNF765
ZNF813
ZNF331
MYADM
PRKCG
CACNG7
CACNG8
NDUFA3
TFPT
PRPF31
CNOT3
LENG1
MBOAT7
TSEN34
TTYH1
LENG8
LENG9
RDH13
EPS8L1
PPP1R12C
TNNT1
TNNI3
DNAAF3
PPP6R1
HSPBP1
BRSK1
KMT5C
COX6B2
TMEM238
ISOC2
C19orf85
ZNF628
NAT14
ZNF579
ZNF784
ZNF580
ZNF581
CCDC106
U2AF2
EPN1
ZNF787
ZNF444
ZSCAN5A
ZNF582
ZNF583
ZNF667
ZFP28
ZNF470
ZNF71
PEG3
ZIM3
ZNF264
AURKC
ZNF805
ZNF543
ZNF304
ZNF548
ZNF17
ZNF549
ZNF550
ZNF416
ZIK1
ZNF530
ZNF134
ZNF211
ZNF551
ZNF671
ZNF776
ZNF552
ZNF587B
ZNF814
ZNF418
ZNF256
ZNF606
ZNF135
ZSCAN18
ZNF329
ZNF274
ZNF544
ZNF8
ZSCAN22
ZNF132
ZNF324
SLC27A5
ZBTB45
TRIM28
CHMP2A
UBE2M
C20orf96
ZCCHC3
SOX12
NRSN2
TRIB3
RBCK1
TBC1D20
CSNK2A1
SRXN1
SLC52A3
RSPO4
PSMF1
NSFL1C
STK35
SNRPB
ZNF343
NOP56
IDH3B
CPXM1
PCED1A
VPS16
PTPRA
UBOX5
FASTKD5
LZTS3
DDRGK1
ITPA
SLC4A11
C20orf194
ATRN
ADAM33
HSPA12B
CENPB
CDC25B
MAVS
PANK2
RNF24
SMOX
PRNP
RASSF2
SLC23A2
TMEM230
PCNA
CDS2
GPCPD1
SHLD1
CHGB
TRMT6
MCM8
CRLS1
FERMT1
BMP2
TMX4
PLCB1
PLCB4
ANKEF1
MKKS
SLX4IP
JAG1
BTBD3
TASP1
ESF1
NDUFAF5
KIF16B
SNRPB2
BFSP1
DSTN
RRBP1
SNX5
OVOL2
MGME1
PET117
KAT14
ZNF133
DZANK1
POLR3F
RBBP9
SEC23B
SMIM26
DTD1
SLC24A3
NAA20
CRNKL1
RALGAPA2
KIZ
XRN2
NXT1
GZF1
NAPB
CST3
SYNDIG1
APMAP
ENTPD6
PYGB
ABHD12
GINS1
NINL
NANP
ZNF337
HM13
ID1
BCL2L1
TPX2
PDRG1
XKR7
CCM2L
TM9SF4
PLAGL2
POFUT1
KIF3B
ASXL1
NOL4L
COMMD7
DNMT3B
MAPRE1
CDK5RAP1
SNTA1
CBFA2T2
NECAB3
E2F1
ZNF341
CHMP4B
RALY
EIF2S2
AHCY
ITCH
DYNLRB1
PIGU
NCOA6
GGT7
ACSS2
GSS
TRPC4AP
EDEM2
PROCR
MMP24OS
MMP24
EIF6
UQCC1
CEP250
ERGIC3
CPNE1
RBM12
NFS1
ROMO1
RBM39
PHF20
SCAND1
EPB41L1
AAR2
DLGAP4
MYL9
TGIF2
RAB5IF
NDRG3
DSN1
SOGA1
SAMHD1
RBL1
RPN2
MANBAL
SRC
BLCAP
NNAT
CTNNBL1
TTI1
RPRD1B
RALGAPB
ACTR5
PPP1R16B
FAM83D
DHX35
TOP1
PLCG1
ZHX3
EMILIN3
CHD6
PTPRT
L3MBTL1
IFT52
MYBL2
TOX2
OSER1
GDAP1L1
FITM2
TTPAL
SERINC3
PKIG
ADA
RIMS4
YWHAB
PABPC1L
TOMM34
STK4
SEMG1
SDC4
SYS1
DBNDD2
PIGT
WFDC2
DNTTIP1
UBE2C
TNNC2
SNX21
ACOT8
ZSWIM3
ZSWIM1
CTSA
PLTP
PCIF1
ZNF335
NCOA5
CD40
SLC35C2
ELMO2
ZNF334
TP53RK
SLC2A10
ZMYND8
NCOA3
SULF2
PREX1
ARFGEF2
CSE1L
STAU1
DDX27
ZNFX1
PTGIS
B4GALT5
SLC9A8
SPATA2
RNF114
UBE2V1
TMEM189
CEBPB
PTPN1
RIPOR3
PARD6B
BCAS4
ADNP
DPM1
MOCS3
KCNG1
ATP9A
SALL4
ZFP64
ZNF217
PFDN4
DOK5
FAM210B
AURKA
CSTF1
RTF2
TFAP2C
BMP7
RAE1
RBM38
RAB22A
VAPB
APCDD1L
STX16
NPEPL1
NELFCD
CTSZ
ATP5F1E
PRELID3B
FAM217B
CDH4
TAF4
LSM14B
PSMA7
SS18L1
MTG2
HRH3
OSBPL2
ADRM1
LAMA5
CABLES2
SLCO4A1
MRGBP
OGFR
COL9A3
TCFL5
DIDO1
GID8
SLC17A9
YTHDF1
ARFGAP1
KCNQ2
EEF1A2
PPDPF
HELZ2
GMEB2
STMN3
RTEL1
ARFRP1
ZGPAT
SLC2A4RG
ZBTB46
TPD52L2
DNAJC5
UCKL1
ZNF512B
PRPF6
TCEA2
RGS19
PCMTD2
RBM11
HSPA13
NRIP1
USP25
CXADR
BTG3
C21orf91
JAM2
GABPA
APP
CYYR1
ADAMTS1
N6AMT1
LTN1
USP16
CCT8
BACH1
TIAM1
SOD1
SCAF4
HUNK
MIS18A
URB1
CFAP298
PAXBP1
IFNAR2
IL10RB
IFNAR1
IFNGR2
TMEM50B
DNAJC28
GART
SON
DONSON
CRYZL1
ITSN1
ATP5PO
SLC5A3
RCAN1
SETD4
CBR1
CBR3
DOP1B
MORC3
CHAF1B
HLCS
PIGP
TTC3
VPS26C
DYRK1A
ERG
ETS2
PSMG1
BRWD1
HMGN1
GET1
SH3BGR
B3GALT5
PCP4
BACE2
MX2
TMPRSS2
PRDM15
C2CD2
ZBTB21
ABCG1
SLC37A1
PDE9A
WDR4
NDUFV3
PKNOX1
CBS
HSF2BP
RRP1B
PDXK
CSTB
RRP1
AGPAT3
TRAPPC10
CFAP410
UBE2G2
SUMO3
PTTG1IP
FAM207A
ADARB1
POFUT2
COL18A1
SLC19A1
PCBP3
COL6A1
COL6A2
SPATC1L
LSS
MCM3AP
YBEY
C21orf58
PCNT
DIP2A
PRMT2
IL17RA
TMEM121B
HDHD5
ADA2
CECR2
ATP6V1E1
BCL2L13
BID
MICAL3
PEX26
USP18
DGCR2
ESS2
SLC25A1
CLTCL1
HIRA
C22orf39
UFD1
CDC45
SEPTIN5
GNB1L
RTL10
TXNRD2
COMT
ARVCF
TANGO2
DGCR8
TRMT2A
ZDHHC8
RTN4R
ZNF74
SCARF2
KLHL22
MED15
PI4KA
SNAP29
CRKL
LZTR1
THAP7
HIC2
UBE2L3
YDJC
SDF2L1
PPIL2
YPEL1
MAPK1
PPM1F
TOP3B
ZNF280B
GNAZ
BCR
ZNF70
MMP11
SMARCB1
DERL3
SLC2A11
MIF
DDTL
DDT
CABIN1
SUSD2
SPECC1L
ADORA2A
GUCD1
LRRC75B
SGSM1
KIAA1671
GRK3
ASPHD2
HPS4
SRRD
TFIP11
TPST2
CRYBB1
CRYBA4
MN1
PITPNB
TTC28
CHEK2
HSCB
CCDC117
XBP1
ZNRF3
EMID1
RHBDD3
EWSR1
GAS2L1
AP1B1
NEFH
THOC5
NIPSNAP1
NF2
ZMAT5
UQCR10
ASCC2
MTMR3
CASTOR1
TBC1D10A
SF3A1
RNF215
SEC14L2
MTFP1
SEC14L4
GAL3ST1
PES1
TCN2
DUSP18
MORC2
SMTN
SELENOM
INPP5J
PLA2G3
RNF185
LIMK2
PATZ1
EIF4ENIF1
SFI1
PISD
PRR14L
DEPDC5
YWHAH
RTCB
FBXO7
LARGE1
HMGXB4
TOM1
HMOX1
MCM5
RASD2
RBFOX2
APOL2
MYH9
TXN2
FOXRED2
EIF3D
IFT27
TST
MPST
KCTD17
MFNG
CARD10
CDC42EP1
GGA1
SH3BP1
PDXP
LGALS1
NOL12
TRIOBP
H1F0
GCAT
ANKRD54
EIF3L
MICALL1
POLR2F
PICK1
PLA2G6
TMEM184B
CSNK1E
KDELR3
DDX17
DMC1
FAM227A
CBY1
TOMM22
JOSD1
GTPBP1
SUN2
DNAL4
CBX6
APOBEC3C
APOBEC3D
APOBEC3F
PDGFB
SYNGR1
TAB1
MGAT3
MIEF1
ATF4
FAM83F
TNRC6B
ADSL
SGSM3
MRTFA
SLC25A17
ST13
XPNPEP3
RBX1
EP300
L3MBTL2
RANGAP1
ZC3H7B
TEF
TOB2
PHF5A
ACO2
POLR3H
PMM1
DESI1
XRCC6
SNU13
CCDC134
SREBF2
TNFRSF13C
CENPM
SEPTIN3
NAGA
PHETA2
SMDT1
NDUFA6
TCF20
RRP7A
POLDIP3
CYB5R3
A4GALT
ARFGAP3
PACSIN2
TTLL1
MCAT
TSPO
TTLL12
SCUBE1
EFCAB6
SULT4A1
PNPLA5
PNPLA3
SAMM50
PARVB
SHISAL1
RTL6
PRR5
ARHGAP8
PHF21B
NUP50
KIAA0930
RIBC2
FBLN1
ATXN10
PPARA
CDPF1
TTC38
GTSE1
TRMU
CELSR1
GRAMD4
CERK
TBC1D22A
TAFA5
BRD1
ZBED4
ALG12
CRELD2
PANX2
TRABD
SELENOO
TUBGCP6
HDAC10
MAPK12
PLXNB2
PPP6R2
SBF1
LMF2
NCAPH2
SCO2
CHKB
MAPK8IP2
ARSA
PLCXD1
GTPBP6
PPP2R3B
CSF2RA
ASMTL
AKAP17A
DHRSX
ZBED1
CD99
GYG2
PRKX
NLGN4X
STS
PNPLA4
ANOS1
TBL1X
GPR143
SHROOM2
WWC3
CLCN4
MID1
HCCS
PRPS2
TMSB4X
RAB9A
OFD1
GPM6B
GEMIN8
FANCB
MOSPD2
PIGA
PIR
CLTRN
CA5B
ZRSR2
AP1S2
GRPR
CTPS2
SYAP1
TXLNG
RBBP7
REPS2
NHS
SCML1
SCML2
CDKL5
PHKA2
ADGRG2
PDHA1
SH3KBP1
BCLAF3
CNKSR2
MBTPS2
SMS
PRDX4
ACOT9
SAT1
APOO
KLHL15
EIF2S3
ZFX
PDK3
PCYT1B
POLA1
TAB3
DMD
TMEM47
PRRG1
DYNLT3
SRPX
RPGR
TSPAN7
MID1IP1
BCOR
ATP6AP2
CXorf38
MED14
USP9X
DDX3X
CASK
MAOA
EFHC2
FUNDC1
KDM6A
KRBOX4
ZNF674
CHST7
SLC9A7
RP2
JADE3
RGN
NDUFB11
RBM10
UBA1
CDK16
USP11
ZNF41
ARAF
SYN1
TIMP1
ELK1
UXT
ZNF182
SLC38A5
FTSJ1
PORCN
EBP
RBM3
WDR13
SUV39H1
HDAC6
PCSK1N
PQBP1
TIMM17B
PIM2
OTUD5
KCND1
GRIPAP1
TFE3
CCDC120
PRAF2
WDR45
GPKOW
PRICKLE3
SYP
CCDC22
PPP1R3F
CLCN5
DGKK
SHROOM4
NUDT11
CENPVL3
GSPT2
MAGED1
GPR173
TSPYL2
KANTR
KDM5C
IQSEC2
SMC1A
HSD17B10
HUWE1
PHF8
FAM120C
WNK3
TSR2
FGD1
GNL3L
MAGED2
TRO
APEX2
FAM104B
MAGEH1
USP51
KLF8
UBQLN2
SPIN3
SPIN2B
FAAH2
SPIN4
ARHGEF9
AMER1
MTMR8
ZC4H2
LAS1L
MSN
HEPH
AR
OPHN1
YIPF6
STARD8
EFNB1
PJA1
FAM155B
EDA
IGBP1
PDZD11
KIF4A
GDPD2
DLG3
SLC7A3
SNX12
FOXO4
MED12
NLGN3
ZMYM3
NONO
ITGB1BP2
TAF1
OGT
NHSL2
ERCC6L
HDAC8
PHKA1
NAP1L2
CHIC1
SLC16A2
RLIM
ABCB7
UPRT
ZDHHC15
PBDC1
MAGEE1
ATRX
MAGT1
COX7B
ATP7A
PGK1
TAF9B
LPAR4
ITM2A
BRWD3
HMGN5
SH3BGRL
HDX
APOOL
ZNF711
CHM
KLHL4
PCDH11X
NAP1L3
DIAPH2
TNMD
TSPAN6
TRMT2B
TMEM35A
CENPI
TIMM8A
GLA
HNRNPH2
ARMCX4
ARMCX1
ARMCX6
ARMCX3
ARMCX2
ZMAT1
TCEAL2
BEX5
TMSB15A
ARMCX5
GPRASP2
BEX1
BEX4
TCEAL8
TCEAL5
BEX2
TCEAL7
TCEAL9
BEX3
TCEAL4
TCEAL1
MORF4L2
PLP1
TMSB15B
SLC25A53
FAM199X
NRK
RADX
TBC1D8B
MORC4
RBM41
NUP62CL
PRPS1
TSC22D3
PSMD10
ATG4A
COL4A6
COL4A5
IRS4
NXT2
KCNE5
ACSL4
TMEM164
AMMECR1
CHRDL1
PAK3
CAPN6
DCX
ALG13
LHFPL1
AMOT
LRCH2
PLS3
KLHL13
WDR44
DOCK11
IL13RA1
PGRMC1
SLC25A43
SLC25A5
CXorf56
UBE2A
NKRF
SEPTIN6
UPF3B
RNF113A
NDUFA1
NKAP
ZBTB33
LAMP2
CUL4B
MCTS1
C1GALT1C1
GRIA3
THOC2
XIAP
STAG2
SMARCA1
OCRL
ZDHHC9
UTP14A
AIFM1
RAB33A
ZNF280C
SLC25A14
RBMX2
ENOX2
IGSF1
STK26
RAP2C
MBNL3
HS6ST2
GPC4
GPC3
CCDC160
PHF6
HPRT1
FAM122B
FAM122C
MOSPD1
RTL8B
RTL8C
RTL8A
SMIM10L2B
ZNF75D
ZNF449
INTS6L
MMGT1
SLC9A6
MAP7D3
HTATSF1
ARHGEF6
RBMX
FGF13
ATP11C
SOX3
LDOC1
SLITRK4
FMR1
AFF2
IDS
TMEM185A
MTM1
MTMR1
CD99L2
VMA21
PRRG3
GABRA3
GABRQ
CETN2
NSDHL
ZNF185
PNMA3
ZNF275
CCNQ
PNCK
SLC6A8
BCAP31
ABCD1
IDH3G
SSR4
PDZD4
L1CAM
ARHGAP4
NAA10
RENBP
HCFC1
IRAK1
MECP2
FLNA
EMD
DNASE1L1
TAZ
ATP6AP1
GDI1
FAM50A
PLXNA3
LAGE3
UBL4A
SLC10A3
FAM3A
G6PD
IKBKG
DKC1
MPP1
FUNDC2
CMC4
BRCC3
VBP1
RAB39B
TMLHE
SPRY3
VAMP7
ZFY
TBL1Y
USP9Y
DDX3Y
UTY
TMSB4Y
NLGN4Y
KDM5D
EIF1AY
MT-CO2
MT-ATP6
MT-CO3
MT-ND3
MT-ND4
MT-ND6
MT-CYB


## Pseudobulk Pearson: leave-one-out over all H1 knockdowns

In [ ]:
!python screen_transfer_h1.py --pseudobulk /content/data/h1_pseudobulk.npz --genes-from /content/data/score_genes.csv \
    --screen k562=/content/data/K562_gwps_normalized_bulk_01.h5ad rpe1=/content/data/rpe1_normalized_bulk_01.h5ad \
             hepg2=/content/data/hepg2_bulk.h5ad jurkat=/content/data/jurkat_bulk.h5ad --out {OUT}/pearson
import pandas as pd
pd.read_csv(f'{OUT}/pearson/screen_transfer_summary.csv').head(25)

## cell-eval on 40 real H1 knockdowns

In [ ]:
!python celleval_screen_h1.py --real /content/data/h1_real_test.h5ad --pseudobulk /content/data/h1_pseudobulk.npz \
    --screens /content/data/K562_gwps_normalized_bulk_01.h5ad /content/data/rpe1_normalized_bulk_01.h5ad \
              /content/data/hepg2_bulk.h5ad /content/data/jurkat_bulk.h5ad --threads 8 --out {OUT}/celleval
pd.read_csv(f'{OUT}/celleval/summary.csv')